In [3]:
#version2 - session_id
# MULTIMODAL PREPROCESSING: EEG + fNIRS 
# Set the paths based on your system before running the code
# you can adjust which subjects to process at the bottom of this cell

import os
import numpy as np
import scipy.io as sio
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, resample
from scipy.interpolate import griddata
from scipy.interpolate import CloughTocher2DInterpolator  # NEW: used by the vectorized spatial_mapping()
from scipy.spatial import Delaunay  # NEW: used by the vectorized spatial_mapping()
import mne
import mne_icalabel
import pywt
import scipy.stats as stats
import pandas as pd
from scipy.interpolate import UnivariateSpline
import matplotlib.pyplot as plt  # NEW: needed for before/after preprocessing plots
import time
import datetime
import json  # NEW: for per-subject metadata.json export (adopted from reviewing a third-party implementation, see chat history)
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
# BASE_DATASET_PATH = r"D:\Research\EEG-fNIRS\project\project\Dataset"
# OUTPUT_PATH = r"D:\Research\EEG-fNIRS\project\project\Results\MULTIMODAL_PREPROCESSED"
# FIX (path update per user's actual folder layout): the user's dataset is laid out as
# DATA\VP001-EEG, DATA\VP001-NIRS, ... (i.e. WITHOUT an intermediate numeric subfolder like
# DATA\001\VP001-EEG that the original code assumed). Paths below updated accordingly; see the
# matching fix in load_and_preprocess_fnirs() and process_subject() further down where
# subject_path/subject_num are removed for the same reason.
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"  # NEW: matches user's actual dataset folder
OUTPUT_PATH = r"F:\preprocessing-history-thesis\test0.0"  # NEW: matches user's actual output folder

# window parameters
EEG_WINDOW_LEN = 3.0      
EEG_STEP = 1.0            # 66% overlap
EEG_EPOCH_TMIN = 0.0
# EEG_EPOCH_TMAX = 8.0
# FIX (Problem 2): the n-back task period is 40 s long (2 s instruction + 40 s task + 20 s rest,
# Shin et al. 2018). The previous value of 8 s only let the sliding window traverse the first
# 8 s of each trial, discarding ~80% of the task period and, more importantly, was being applied
# on top of STIMULUS-level markers (every ~2 s) rather than TRIAL-level markers, which produced
# massive window overlap across different stimuli instead of covering the full cognitive-load trial.
EEG_EPOCH_TMAX = 40.0      # NEW: cover the full 40 s task period of a single n-back trial

# FNIRS_WINDOW_LEN = 10.0
# CHANGED (EXPERIMENTAL -- 11-segment fNIRS windowing, per chat history): matches the TSFNet
# paper's actual approach more closely than the single continuous 10s window used previously --
# instead of one 10s fNIRS window per EEG window, 11 overlapping 3s segments (stepped by
# FNIRS_STEP=1s each) are extracted and stacked, giving the model an explicit per-segment temporal
# axis (matching Table 1's fNIRS branch input) rather than one pre-averaged continuous block.
FNIRS_WINDOW_LEN = 3.0    # CHANGED: 10.0 -> 3.0 (length of EACH of the 11 segments, not the whole span)
FNIRS_NUM_SEGMENTS = 11   # NEW: number of overlapping 3s segments per EEG window, per the paper's approach
FNIRS_STEP = 1.0          # 90% overlap

# NEW (isolated baseline-correction test, per chat history): the paper's Sec. 2.2 also specifies
# "baseline correction was performed by subtracting the average signal within the -5 to -2 s
# interval before each trial" -- a step that was previously only ever tested BUNDLED together
# with the paper's bandpass filter (Run C/E, both rejected), so its effect was never isolated.
# Testing it alone here, with the current (already-validated) 0.2 Hz lowpass filter kept UNCHANGED.
BASELINE_TMIN = -5.0  # NEW: seconds before trial onset (start of baseline window)
BASELINE_TMAX = -2.0  # NEW: seconds before trial onset (end of baseline window)
FNIRS_EPOCH_TMIN = 0.0
FNIRS_EPOCH_TMAX = 18.0   # bacause of the hemodynamic delay
# NOTE: FNIRS_EPOCH_TMAX is no longer used directly to bound the slide range (the EEG trial length
# now governs how many windows are produced, see extract_synchronized_windows below), but it is left
# untouched here exactly as before so no other part of the code/structure changes.

SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0      

# NEW (Problem 1 fix support): explicit trial-level marker maps based on the official dataset
# documentation (Shin et al. 2018, Dataset_description_MATLAB.pdf):
#   EEG  mrk_nback.event.desc : 112 = 0-back session, 128 = 2-back session, 144 = 3-back session
#   fNIRS mrk_nback.event.desc:   7 = 0-back session,   8 = 2-back session,   9 = 3-back session
# These TRIAL-onset codes (one per ~40 s task period) replace the old STIMULUS-level codes
# (16/48/64/80/96 for EEG; 1/2/3/4/5 placeholders) that fired every ~2 s within a trial.
EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}      # NEW: EEG session/trial-onset codes -> class
FNIRS_TRIAL_LABEL_MAP = {7: 0, 8: 1, 9: 2}          # NEW: fNIRS session/trial-onset codes -> class


# LOAD EEG DATA
def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    
    print(f"Loading EEG from: {cnt_path}")
    
    # Load signal
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]
    cnt_key = real_cnt_keys[0]
    cnt_struct = mat_cnt[cnt_key]
    
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    
    # making sure the sampling rate of 200
    if sfreq != SFREQ_EEG:
        print(f"Resampling from {sfreq} to {SFREQ_EEG} Hz")
        from scipy.signal import resample
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    
    # EOG channels
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    
    # NOTE: this generic standard_1005 template montage is kept ONLY for MNE-internal purposes that
    # need SOME position estimate for every channel (e.g. interpolate_bads()'s spherical-spline
    # interpolation in preprocess_eeg()). It is NOT used for the spatial_mapping() grid anymore --
    # get_eeg_mnt() (further below) uses the REAL, dataset-provided positions from mnt_nback.mat
    # (loaded at the end of this function) for that purpose instead.
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')
    
    # Load markers 
    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    real_mrk_keys = [k for k in mat_mrk.keys() if not k.startswith('__')]
    mrk_key = real_mrk_keys[0]
    mrk_struct = mat_mrk[mrk_key]
    
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("Could not find 'time' or 'pos' in marker struct")
    
    # Event extraction
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    elif hasattr(mrk_struct, 'y'):
        y = mrk_struct.y
        if y.ndim > 1:
            if y.shape[0] == 1 or y.shape[1] == 1:
                event_codes = y.flatten()
            else:
                event_codes = np.argmax(y, axis=0) + 1
        else:
            event_codes = y.flatten()
    else:
        event_codes = event_positions.copy()
    
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes
    
    print(f"EEG loaded: {raw.n_times} samples, {sfreq} Hz, {len(events)} markers")
    print(f"unique markers: {np.unique(event_codes)}")

    # NEW: load the EEG mnt_nback.mat file -- REAL electrode 3D positions from this dataset,
    # confirmed via user-provided inspection script output: same struct format as the fNIRS
    # mnt_nback.mat already used in load_and_preprocess_fnirs (fields ['x','y','pos_3d','clab'],
    # pos_3d shape (3, 30), clab shape (30,)). This CORRECTS an earlier (wrong) assumption that EEG
    # had no equivalent .mat position file and had to fall back on MNE's generic standard_1005
    # template positions (raw.set_montage() below, which is KEPT unchanged -- it is still needed
    # for interpolate_bads()'s spherical-spline interpolation, an unrelated MNE-internal use). For
    # the azimuthal-equidistant spatial_mapping() grid, get_eeg_mnt() (defined further below) now
    # uses THIS real, dataset-provided mnt_nback.mat instead, exactly mirroring how fNIRS's own
    # mnt_nback.mat is already used for its spatial grid.
    mnt_path = os.path.join(eeg_folder, 'mnt_nback.mat')  # NEW
    mat_eeg_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW
    real_eeg_mnt_keys = [k for k in mat_eeg_mnt.keys() if not k.startswith('__')]  # NEW
    eeg_mnt_struct = mat_eeg_mnt[real_eeg_mnt_keys[0]]  # NEW: has .pos_3d (3,30) and .clab (30,)
    print(f"EEG mnt_nback.mat loaded: pos_3d shape {np.asarray(eeg_mnt_struct.pos_3d).shape}")  # NEW

    # return raw, events, sfreq
    return raw, events, sfreq, eeg_mnt_struct  # CHANGED: also return the real electrode-position struct


# LOAD AND PREPROCESS FNIRS DATA
def load_and_preprocess_fnirs(subject_id):
    # subject_num = subject_id[2:]
    # subject_path = os.path.join(BASE_DATASET_PATH, subject_num)
    # fnirs_folder = os.path.join(subject_path, f"{subject_id}-NIRS")
    # FIX: removed the intermediate numeric subfolder (e.g. "001") which the user's actual dataset
    # layout does not have; fNIRS data sits directly under BASE_DATASET_PATH\VPxxx-NIRS.
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")  # NEW: flat folder structure
    
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(fnirs_folder, 'mrk_nback.mat')  # NEW: fNIRS trial markers needed for the quality-check plot and (optionally) downstream use
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fNIRS file not found: {cnt_path}")
    
    print(f"Loading raw fNIRS from: {cnt_path}")
    
    # cnt_data = loadmat(cnt_path)
    # cnt = cnt_data['cnt_nback'][0, 0]
    #
    # mnt_data = loadmat(mnt_path)
    # mnt = mnt_data['mnt_nback'][0, 0]
    # FIX (Problem 1): the lines above used the default scipy.io.loadmat() call (struct_as_record=True),
    # while the rest of the pipeline (load_eeg_data) consistently uses
    # struct_as_record=False, squeeze_me=True. Mixing the two access styles is fragile: under the
    # default mode, struct fields are accessed via numpy structured-array indexing (e.g. cnt['oxy'][0,0]['x']),
    # while under struct_as_record=False they are accessed as plain Python attributes (e.g. cnt.oxy.x).
    # Verified against the actual VP001 .mat structure (inspect_data.py output): cnt_nback has fields
    # ['oxy','deoxy'], each of which is itself a struct with field 'x' holding the (n_samples, 36) signal,
    # and mnt_nback has fields ['x','y','pos_3d','clab'] holding (36,) coordinate vectors. Using the
    # unified squeeze_me=True / struct_as_record=False loader removes the [0,0] nested-indexing style and
    # avoids a silent shape mismatch in spatial_mapping() if the indexing assumption is ever wrong.
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)  # NEW: unified loader (matches load_eeg_data)
    real_cnt_keys = [k for k in mat_cnt.keys() if not k.startswith('__')]     # NEW
    cnt = mat_cnt[real_cnt_keys[0]]                                          # NEW: cnt is now a MATLAB-struct object with .oxy / .deoxy attributes

    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)  # NEW: unified loader (matches load_eeg_data)
    real_mnt_keys = [k for k in mat_mnt.keys() if not k.startswith('__')]     # NEW
    mnt = mat_mnt[real_mnt_keys[0]]                                          # NEW: mnt is now a MATLAB-struct object with .x / .y attributes

    # Load fNIRS trial markers (NEW: needed only for the before/after QC plot at the end of this
    # function; segmentation itself happens later using EEG_TRIAL_LABEL_MAP / FNIRS_TRIAL_LABEL_MAP
    # inside extract_synchronized_windows, which keeps this function's return signature unchanged).
    fnirs_events_time = None  # NEW
    fnirs_events_code = None  # NEW
    if os.path.exists(mrk_path):  # NEW
        mat_fmrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)  # NEW
        real_fmrk_keys = [k for k in mat_fmrk.keys() if not k.startswith('__')]    # NEW
        fmrk = mat_fmrk[real_fmrk_keys[0]]                                         # NEW
        if hasattr(fmrk, 'time') and hasattr(fmrk, 'event') and hasattr(fmrk.event, 'desc'):  # NEW
            fnirs_events_time = np.asarray(fmrk.time).flatten()                     # NEW
            fnirs_events_code = np.asarray(fmrk.event.desc).flatten()               # NEW
    
    # Extract actual sampling rate (making sure it's 10Hz)
    # sfreq_fnirs_raw = float(cnt['fs'][0, 0]) if 'fs' in cnt.dtype.names else 10.0
    # FIX (Problem 1, continued): 'fs' is a field of cnt.oxy / cnt.deoxy (each a sub-struct), not of
    # cnt itself (verified: cnt_nback.oxy.fs == 10, cnt_nback.deoxy.fs == 10 in the inspected VP001 file).
    # The original line would raise an AttributeError under the new loader (cnt has no .dtype/.fs), so
    # it must read fs from the oxy sub-struct.
    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0  # NEW
    print(f"raw fNIRS sampling rate: {sfreq_fnirs_raw} Hz")
    
    # Extract raw oxy/deoxy
    # if 'oxy' in cnt.dtype.names:
    #     oxy_raw = cnt['oxy'][0, 0]['x']
    #     deoxy_raw = cnt['deoxy'][0, 0]['x']
    # else:
    #     oxy_raw = cnt['oxy']
    #     deoxy_raw = cnt['deoxy']
    # FIX (Problem 1, continued): with struct_as_record=False, squeeze_me=True, the signal matrix is
    # accessed as cnt.oxy.x / cnt.deoxy.x (plain attribute access), matching the (n_samples, 36) shape
    # confirmed in inspect_data.py for VP001 (21178, 36).
    oxy_raw = np.asarray(cnt.oxy.x, dtype=np.float64)     # NEW
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)  # NEW
    
    print(f"Raw fNIRS: oxy {oxy_raw.shape}, deoxy {deoxy_raw.shape}")
    
    # Convert to HbO/HbR
    # print("Converting to HbO/HbR...")
    # hbo_raw, hbr_raw = calculate_hbo_hbr(oxy_raw, deoxy_raw)
    # CHANGED (EXPERIMENTAL -- Beer-Lambert removal test, per chat history): the official dataset
    # Data Descriptor (Shin et al. 2018) explicitly states "cnt files contain deoxy/oxy-hemoglobin
    # data as separate fields... The raw NIRS optical intensity data converted to deoxy-(HbR) and
    # oxy hemoglobin (HbO) concentration changes... using the modified Beer-Lambert law" BEFORE
    # distribution -- confirmed further by the dataset description PDF itself, which explicitly
    # labels these fields "oxy hemoglobin data" / "deoxy hemoglobin data" (not raw optical density).
    # oxy_raw/deoxy_raw are therefore already ΔHbO/ΔHbR; calling calculate_hbo_hbr() on them
    # re-applies Beer-Lambert to already-converted concentration data, which is physically incorrect.
    # calculate_hbo_hbr() itself is left defined-but-unused below (not deleted), per the instruction
    # not to touch other parts of the file.
    print("Using oxy/deoxy fields directly as HbO/HbR (Beer-Lambert already applied by dataset authors; see Shin et al. 2018 Data Descriptor)...")  # NEW
    hbo_raw, hbr_raw = oxy_raw, deoxy_raw  # NEW: no conversion -- oxy_raw IS HbO, deoxy_raw IS HbR
    
    # Wavelet denoising 
    # print("Wavelet denoising...")
    # hbo_wv = wavelet_denoising(hbo_raw)
    # hbr_wv = wavelet_denoising(hbr_raw)
    #
    # MARA on full signal
    # print("MARA...")
    # hbo_mara = apply_mara(hbo_wv, fs=sfreq_fnirs_raw)
    # hbr_mara = apply_mara(hbr_wv, fs=sfreq_fnirs_raw)
    #
    # Low-pass filter 
    # FNIRS_LOWPASS_CUTOFF_HZ = 0.2
    # hbo_filtered = lowpass_filter(hbo_mara, fs=sfreq_fnirs_raw, cutoff=FNIRS_LOWPASS_CUTOFF_HZ)
    # hbr_filtered = lowpass_filter(hbr_mara, fs=sfreq_fnirs_raw, cutoff=FNIRS_LOWPASS_CUTOFF_HZ)
    # CHANGED (EXPERIMENTAL -- isolated fNIRS-filtering-removal test, Priority 2, per chat history):
    # Ding et al. (2026, Information Fusion, "MSTFDN"), using this SAME dataset, applies NO
    # wavelet denoising, NO MARA, and NO separate lowpass/bandpass filtering to fNIRS at all --
    # their stated rationale: "since the fNIRS branch uses convolution kernels with a large size
    # and receptive field (large size temporal convolution is equivalent to low-pass filtering in
    # the frequency domain), the model naturally focuses on low-frequency components during
    # feature learning. Therefore, we did not perform separate filtering preprocessing." Their
    # fNIRS pipeline is simply: Beer-Lambert (already done by dataset authors) -> baseline
    # correction -> z-score normalization. This test skips wavelet_denoising()/apply_mara()/
    # lowpass_filter() entirely (functions left defined-but-unused, not deleted) to test this
    # simplification hypothesis directly against our own hand-designed filtering pipeline, on the
    # SAME CONFIRMED baseline-corrected Run H pipeline (isolated from Priority 1's baseline-window
    # change and the still-unconfirmed EEG filter change).
    print("Skipping wavelet/MARA/lowpass filtering (MSTFDN-style: relying on the model's own large-kernel convolutions for implicit smoothing)...")  # NEW
    hbo_filtered, hbr_filtered = hbo_raw, hbr_raw  # NEW: pass-through, no denoising/artifact-removal/filtering applied
    
    # dynamic downsampling to 10 Hz
    print(f"Resampling from {sfreq_fnirs_raw} Hz to {SFREQ_FNIRS} Hz...")
    from scipy.signal import resample
    n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
    hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
    hbr_resampled = resample(hbr_filtered, n_samples, axis=0)
    
    # spatial mapping to 16x16
    print("Spatial mapping to 16x16...")
    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    
    # Stack as channels: (n_samples, 16, 16, 2)
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)
    
    print(f"fNIRS preprocessed: {fnirs_data.shape}")

    # NEW (user request): save a before/after preprocessing QC plot for fNIRS (one representative
    # channel, raw HbO vs fully preprocessed/resampled HbO). Saved per-subject next to that subject's
    # output folder. Does not alter any returned value or downstream logic.
    try:  # NEW
        qc_channel = 0  # NEW: first fNIRS channel, arbitrary but fixed for reproducibility
        t_raw = np.arange(oxy_raw.shape[0]) / sfreq_fnirs_raw  # NEW
        t_proc = np.arange(hbo_resampled.shape[0]) / SFREQ_FNIRS  # NEW
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)  # NEW
        axes[0].plot(t_raw, oxy_raw[:, qc_channel], color='tab:red', linewidth=0.6)  # NEW
        axes[0].set_title(f"{subject_id} - fNIRS RAW HbO (channel {qc_channel}, before preprocessing)")  # NEW
        axes[0].set_xlabel("Time (s)")  # NEW
        axes[0].set_ylabel("HbO (raw units)")  # NEW
        axes[1].plot(t_proc, hbo_resampled[:, qc_channel], color='tab:blue', linewidth=0.8)  # NEW
        axes[1].set_title(f"{subject_id} - fNIRS PREPROCESSED HbO (channel {qc_channel}, after wavelet+MARA+lowpass+resample)")  # NEW
        axes[1].set_xlabel("Time (s)")  # NEW
        axes[1].set_ylabel("HbO (mmol/L)")  # NEW
        plt.tight_layout()  # NEW
        qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
        os.makedirs(qc_dir, exist_ok=True)  # NEW
        fig.savefig(os.path.join(qc_dir, f"{subject_id}_fnirs_before_after.png"), dpi=150)  # NEW
        plt.close(fig)  # NEW
        print(f"Saved fNIRS before/after QC plot for {subject_id}")  # NEW
    except Exception as e:  # NEW
        print(f"Could not save fNIRS QC plot for {subject_id}: {e}")  # NEW
    
    # return fnirs_data
    # CHANGED (CRITICAL SYNCHRONIZATION FIX): diagnostic testing (see chat history) directly compared
    # EEG's own trial markers (codes 112/128/144) against fNIRS's own trial markers (codes 7/8/9),
    # both loaded independently from their respective mrk_nback.mat files. This revealed that the
    # two devices do NOT share a common time origin: within each of the 3 class "sessions" the two
    # devices agree almost exactly (delta ~0.1-0.3s, pure jitter), but ACROSS session boundaries a
    # near-constant ~46-47s jump appears (e.g. VP001: session 1 delta=16.6s, session 2 delta=62.7s,
    # session 3 delta=110.0s), consistently reproduced across all 5 subjects tested. Since
    # extract_synchronized_windows() previously computed the fNIRS window position purely by
    # converting EEG's own event time to an fNIRS sample index (assuming a shared t=0), this means
    # for the 2-back and 3-back sessions the fNIRS window was being read from a position roughly
    # 46-93 seconds away from where it should be -- i.e. NOT synchronized with the EEG window it was
    # being paired with. fnirs_events_time and fnirs_events_code (already loaded above, previously
    # used only for the QC plot) are now also returned so extract_synchronized_windows() can use
    # fNIRS's OWN trial markers as the anchor for fNIRS windowing, instead of a value derived from
    # EEG's clock.
    return fnirs_data, fnirs_events_time, fnirs_events_code  # CHANGED: also return fNIRS's own trial markers

# fNIRS HELPER FUNCTIONS
def calculate_hbo_hbr(oxy, deoxy):
    lambda1, lambda2 = 760, 850
    L = 3.0
    
    def get_dpf(lmbda, age=25):
        alpha, beta, gamma = 223.3, 0.05624, 0.8493
        delta, eps, zeta = -5.723e-7, 0.001245, -0.9025
        return alpha + beta * (age ** gamma) + delta * (lmbda ** 3) + eps * (lmbda ** 2) + zeta * lmbda
    
    dpf1 = get_dpf(lambda1)
    dpf2 = get_dpf(lambda2)
    
    e1_hbo, e1_hbr = 0.586, 1.54852
    e2_hbo, e2_hbr = 1.058, 0.69132
    
    denom = L * (e1_hbr * e2_hbo - e2_hbr * e1_hbo)
    hbo = (e1_hbr * (deoxy / dpf2) - e2_hbr * (oxy / dpf1)) / denom
    hbr = (e1_hbo * (oxy / dpf1) - e2_hbo * (deoxy / dpf2)) / denom
    
    return hbo, hbr

def wavelet_denoising(data, alpha=0.1, wavelet='db5'):
    cleaned = np.zeros_like(data)
    n_samples, n_channels = data.shape
    
    for ch in range(n_channels):
        signal = data[:, ch]
        level = min(int(np.floor(np.log2(n_samples))), 6)
        coeffs = pywt.wavedec(signal, wavelet, level=level)
        new_coeffs = [coeffs[0]]
        for i in range(1, len(coeffs)):
            d_coeffs = coeffs[i].copy()
            sigma = np.median(np.abs(d_coeffs - np.median(d_coeffs))) / 0.6745
            if sigma > 0:
                threshold = sigma * stats.norm.ppf(1 - alpha / 2)
                d_coeffs[np.abs(d_coeffs) > threshold] = 0
            new_coeffs.append(d_coeffs)
        cleaned[:, ch] = pywt.waverec(new_coeffs, wavelet)[:n_samples]
    
    return cleaned

def apply_mara(data, fs=10.0, window_size=41, threshold_factor=1.8):
    n_samples, n_channels = data.shape
    corrected = np.zeros_like(data)
    edge_len = int(0.5 * fs)
    
    for ch in range(n_channels):
        signal = data[:, ch].copy()
        msd = pd.Series(signal).rolling(window=window_size, center=True, min_periods=1).std().to_numpy()
        threshold = np.median(msd) + (threshold_factor * np.std(msd))
        ma_mask = (msd > threshold).astype(int)
        
        diff_mask = np.diff(ma_mask, prepend=0, append=0)
        starts = np.where(diff_mask == 1)[0]
        ends = np.where(diff_mask == -1)[0]
        indices = sorted(list(set([0, n_samples] + list(starts) + list(ends))))
        segments = [signal[indices[i]:indices[i+1]].copy() for i in range(len(indices) - 1)]
        
        if not segments:
            corrected[:, ch] = signal
            continue
        
        final_parts = [segments[0]]
        for i in range(1, len(segments)):
            prev_seg = final_parts[-1]
            curr_seg = segments[i]
            val_prev = np.median(prev_seg[-edge_len:]) if len(prev_seg) >= edge_len else prev_seg[-1]
            val_curr = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
            shift = val_prev - val_curr
            
            if ma_mask[indices[i]] == 1:
                x = np.arange(len(curr_seg))
                s_val = max(np.var(curr_seg) * len(curr_seg), 1e-6)
                try:
                    spline = UnivariateSpline(x, curr_seg, s=s_val)
                    curr_seg = curr_seg - spline(x)
                    val_curr_new = np.median(curr_seg[:edge_len]) if len(curr_seg) >= edge_len else curr_seg[0]
                    curr_seg += (val_prev - val_curr_new)
                except:
                    curr_seg += shift
            else:
                if abs(shift) > 2 * np.std(msd):
                    curr_seg += shift
            final_parts.append(curr_seg)
        
        full_signal = np.concatenate(final_parts)[:n_samples]
        t_idx = np.arange(n_samples)
        original_trend = np.polyval(np.polyfit(t_idx, data[:, ch], 1), t_idx)
        corrected_trend = np.polyval(np.polyfit(t_idx, full_signal, 1), t_idx)
        full_signal += (original_trend - corrected_trend)
        corrected[:, ch] = full_signal
    
    return corrected

def lowpass_filter(data, fs=10.0, cutoff=0.15):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(3, normal_cutoff, btype='low')
    filtered = np.zeros_like(data)
    for ch in range(data.shape[1]):
        filtered[:, ch] = filtfilt(b, a, data[:, ch])
    return filtered

def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    # x_coords = np.asarray(mnt.x).flatten()
    # y_coords = np.asarray(mnt.y).flatten()
    # FIX (projection accuracy): mnt.x / mnt.y are pre-computed 2D coordinates from the BBCI
    # toolbox's own projection, which is not necessarily the azimuthal equidistant projection
    # described in the TSFNet reference paper (Zhang, Yin, Yuan, Sensors 2025). Using the raw 3D
    # channel positions (mnt.pos_3d, shape (3, n_channels), confirmed present in the fNIRS mnt_nback
    # struct via inspect_data.py) and projecting them with a genuine azimuthal equidistant
    # projection (angular distance from the vertex is preserved: theta = angle from north pole,
    # phi = azimuthal angle) more faithfully reproduces the paper's methodology than reusing the
    # toolbox's built-in 2D coordinates.
    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)  # NEW: shape (3, n_channels)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]  # NEW
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)  # NEW
    r_safe = np.where(r > 0, r, 1.0)  # NEW: avoid division by zero for any degenerate channel position
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))  # NEW: angle from the vertex (pole)
    phi = np.arctan2(y3d, x3d)  # NEW: azimuthal angle
    x_coords = theta * np.cos(phi)  # NEW: azimuthal equidistant projection
    y_coords = theta * np.sin(phi)  # NEW
    
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, 
                               min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    
    # mapped = np.zeros((n_samples, 16, 16), dtype=np.float32)
    # for t in range(n_samples):
    #     grid_z = griddata(points, data[t, :], (grid_x, grid_y), method='cubic', fill_value=0)
    #     mapped[t] = grid_z
    # PERF FIX: the loop above calls griddata() once per time sample, re-computing the Delaunay
    # triangulation of `points` from scratch every single time -- even though `points` (electrode
    # positions) NEVER changes across time samples for a given subject; only the values (data[t,:])
    # change. Confirmed via VP001/VP002 real runs, EEG spatial mapping (394930 / 395238 samples at
    # 200 Hz) was the dominant cost of the ~28-32 min per-subject runtime. griddata's 'cubic' 2D
    # path internally builds scipy.interpolate.CloughTocher2DInterpolator(points, values, ...),
    # which officially supports (a) being given a pre-built scipy.spatial.Delaunay triangulation
    # instead of raw points, and (b) vector-valued `values` of shape (n_points, ...trailing_dims),
    # interpolating all trailing "columns" (here: all time samples) in one batched call. Verified
    # numerically bit-identical (max abs diff = 0.0) against the old per-sample loop on synthetic
    # data of the same shape before applying this change (see chat history for the verification
    # script). This is a pure performance refactor -- the interpolation math is unchanged.
    tri = Delaunay(points)  # NEW: triangulation depends only on `points`, computed ONCE instead of n_samples times
    values = data.T  # NEW: (n_samples, n_channels) -> (n_channels, n_samples); CloughTocher2DInterpolator
                      # NEW: requires values.shape[0] == points.shape[0] (== n_channels here)
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)  # NEW: gradient estimation + patch setup done once, for all time samples together
    result = interp((grid_x, grid_y))  # NEW: shape (16, 16, n_samples) -- one batched evaluation instead of n_samples separate ones
    mapped = np.transpose(result, (2, 0, 1)).astype(np.float32)  # NEW: (16, 16, n_samples) -> (n_samples, 16, 16), matching the original return shape/dtype exactly
    
    return mapped

# NEW (EEG spatial mapping support): spatial_mapping() above was written for fNIRS, which reads
# 3D channel positions from mnt.pos_3d (a field already present in the dataset's mnt_nback.mat
# struct). EEG has no equivalent .mat mnt struct -- its electrode positions instead live inside
# the MNE montage set in load_eeg_data() via raw.set_montage('standard_1005', match_case=False,
# on_missing='ignore'). _EEGMontageAdapter + get_eeg_mnt() below extract those positions, in the
# SAME channel order as raw.ch_names (so they line up 1:1 with data columns passed to
# spatial_mapping), into a lightweight object exposing .pos_3d with shape (3, n_channels) --
# exactly the interface spatial_mapping() already expects -- so spatial_mapping() itself does not
# need to change at all and is reused UNCHANGED for EEG, with the same azimuthal equidistant
# projection methodology already used for fNIRS.
# NEW (EEG spatial mapping support): spatial_mapping() above was written for fNIRS, which reads
# 3D channel positions from mnt.pos_3d (a field present in the dataset's fNIRS mnt_nback.mat
# struct). The EEG folder has its OWN mnt_nback.mat with the exact same format (pos_3d shape
# (3, 30), clab shape (30,)), confirmed via direct inspection of VP001-EEG/mnt_nback.mat -- so EEG
# positions come from the REAL, dataset-provided file, not from MNE's generic standard_1005
# template (an earlier, incorrect assumption; see the comment above raw.set_montage() in
# load_eeg_data(), where that template montage is kept only for interpolate_bads()'s internal use).
# _EEGMontageAdapter + get_eeg_mnt() below select, from the full 30-channel mnt_nback.mat struct,
# only the positions of the channels that SURVIVED preprocessing (raw.ch_names, after
# VEOG/HEOG/TP9 are dropped in preprocess_eeg), matched BY NAME against mnt_nback.mat's own clab
# list (channel order in cnt_nback.mat and mnt_nback.mat is not guaranteed to be identical, so
# matching by name rather than by raw positional index is required for correctness). The result is
# a lightweight object exposing .pos_3d with shape (3, n_channels) -- exactly the interface
# spatial_mapping() already expects -- so spatial_mapping() itself needs no changes at all.
class _EEGMontageAdapter:  # NEW
    """Minimal stand-in for the fNIRS .mat mnt struct, exposing only the .pos_3d attribute that
    spatial_mapping() reads."""
    def __init__(self, pos_3d):  # NEW
        self.pos_3d = pos_3d  # NEW: shape (3, n_channels), matching mnt.pos_3d's shape for fNIRS

def get_eeg_mnt(raw, eeg_mnt_struct):  # CHANGED: now takes the real mnt_nback.mat struct (loaded once in load_eeg_data) instead of deriving positions from raw's MNE montage
    """
    Builds an _EEGMontageAdapter (pos_3d shape (3, n_channels)) from the REAL electrode 3D
    positions in the dataset's own EEG mnt_nback.mat (eeg_mnt_struct.pos_3d / .clab), selecting
    and ordering them to match raw.ch_names exactly. Must be called AFTER any channel drops
    (VEOG/HEOG/TP9 in preprocess_eeg) so the returned position order matches the channel order of
    the data that will be passed to spatial_mapping().
    """
    full_clab = [str(c).strip() for c in np.asarray(eeg_mnt_struct.clab).flatten()]  # NEW: e.g. 30 names, same cleanup style as cnt_struct.clab in load_eeg_data
    full_pos_3d = np.asarray(eeg_mnt_struct.pos_3d, dtype=np.float64)  # NEW: shape (3, 30)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}  # NEW
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}  # NEW: case-insensitive fallback, matches match_case=False used elsewhere in this pipeline

    pos_list = []  # NEW
    missing = []  # NEW
    for ch in raw.ch_names:  # NEW
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))  # NEW
        if idx is None:  # NEW: channel present in raw but not found in mnt_nback.mat's clab list
            missing.append(ch)  # NEW
            pos_list.append([0.0, 0.0, 0.0])  # NEW: origin fallback (theta=0 -> maps to grid center)
        else:  # NEW
            pos_list.append(full_pos_3d[:, idx])  # NEW

    if missing:  # NEW
        print(f"  WARNING: {len(missing)} EEG channel(s) not found in mnt_nback.mat's clab "  # NEW
              f"(spatial_mapping will place them at the grid center): {missing}")  # NEW

    pos_3d = np.array(pos_list, dtype=np.float64).T  # NEW: shape (3, n_channels)
    return _EEGMontageAdapter(pos_3d)  # NEW

# PREPROCESS EEG
def preprocess_eeg(raw, events):
    print("Preprocessing EEG...")

    # NEW (user request): keep a copy of the raw signal (one representative channel) for the
    # before/after preprocessing QC plot saved at the end of this function.
    raw_before_qc = raw.copy()  # NEW
    
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks='eeg', verbose=False)
    raw_filtered.filter(l_freq=1.0, h_freq=40.0, method='fir', verbose=False)
    
    # Detect bad channels
    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method='fir', verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ['VEOG', 'HEOG', 'TP9']:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)
    
    if bad_channels:
        print(f"Bad channels: {bad_channels}")
        raw_filtered.info['bads'] = bad_channels
    
    # NEW (fix support): remember which channels were flagged as bad BEFORE interpolate_bads()
    # resets info['bads']. This list is needed further below to correctly compute the ICA rank,
    # since interpolate_bads(reset_bads=True) clears info['bads'] and the original code's
    # exclude='bads' filter therefore silently included the interpolated channels in the ICA fit.
    interpolated_channels = list(raw_filtered.info['bads'])  # NEW
    
    if raw_filtered.info['bads']:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    
    raw_filtered.set_eeg_reference('average', projection=False, verbose=False)
    
    # ICA
    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude='bads')
    # corrected_rank = max(1, len(picks_for_ica) - 1)
    # FIX (rank problem): interpolate_bads(reset_bads=True) above already cleared info['bads'],
    # so exclude='bads' here has NOTHING to exclude and picks_for_ica silently includes every
    # interpolated channel. Interpolated channels are linear combinations of their neighbours and
    # add NO new degrees of freedom to the data, so the true data rank is
    # (n_good_channels - 1 for CAR - n_interpolated_channels), not just (n_channels - 1).
    # Asking ICA (via n_components=corrected_rank) for more components than the true rank forces
    # the algorithm to spread a single real source (e.g. the eye-blink dipole) across multiple
    # weak, "diluted" components, none of which individually scores high enough in ICLabel
    # (whether by argmax or by the probability threshold added above) to be flagged as
    # 'eye blink'. This is very likely the main reason eye-blink artifacts survive in this
    # pipeline's output while code 1 (which never interpolates channels) does not have this issue.

###########

    # corrected_rank = max(1, len(picks_for_ica) - 1 - len(interpolated_channels))
    # print(f"ICA rank: {corrected_rank} (picks={len(picks_for_ica)}, interpolated={len(interpolated_channels)})")
    # ica = mne.preprocessing.ICA(n_components=corrected_rank, method='infomax',
    #                             fit_params=dict(extended=True), random_state=42, verbose=False)
    # FIX (rank problem, cleaner solution): the manual rank correction above (subtracting the number
    # of interpolated channels) is a reasonable workaround, but n_components=0.99 solves the same
    # problem more robustly and automatically: instead of a FIXED component count, ICA's internal PCA
    # step keeps only as many components as needed to explain 99% of the data's variance. Interpolated
    # channels contribute (near-)zero additional variance since they are linear combinations of their
    # neighbours, so PCA naturally discards that redundant dimensionality without needing to manually
    # count and subtract interpolated channels. This avoids asking ICA for more components than the
    # true data rank, which is what caused the eye-blink source to be diluted across several weak,
    # under-scored components in the original pipeline (see comment above find_bads_eog for the full
    # rationale of that issue).
    # method='picard' (Ablin, Cardoso & Gramfort, 2018, IEEE Signal Processing Letters) is mathematically
    # equivalent to extended infomax but converges faster and more reliably; it is the ICA algorithm
    # used in the reference implementation (code 1) and is increasingly the default recommendation in
    # recent BCI/EEG preprocessing literature.
    print(f"ICA: using variance-based component selection (99% explained variance), "  # NEW
          f"picks={len(picks_for_ica)}, interpolated={len(interpolated_channels)}")     # NEW
    ica = mne.preprocessing.ICA(n_components=0.99, method='picard',                     # NEW
                                fit_params=dict(ortho=False, extended=True),            # NEW
                                random_state=42, verbose=False)                          # NEW
    ica.fit(raw_filtered, picks=picks_for_ica)


#########    

####
    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method='iclabel')

    ARTIFACT_CLASSES_IDX = {  # NEW: standard ICLabel class order (verified from ica.labels_scores_ shape=(n,7))
        'brain': 0, 'muscle artifact': 1, 'eye blink': 2,
        'heart beat': 3, 'line noise': 4, 'channel noise': 5, 'other': 6
    }
    ARTIFACT_TARGETS = ['eye blink', 'eye movement', 'muscle artifact', 'line noise']  # NEW
    PROB_THRESHOLD = 0.30  # NEW

    # Original argmax-based selection (unchanged logic)
    artifact_idx_argmax = [idx for idx, label in enumerate(ic_labels['labels'])
                           if label in ARTIFACT_TARGETS]  # CHANGED: renamed

    # NEW: probability-threshold based selection using ica.labels_scores_ (shape: n_components x 7)
    artifact_idx_prob = []  # NEW
    proba_matrix = ica.labels_scores_  # NEW: verified shape (27, 7) for this dataset
    for comp_idx in range(proba_matrix.shape[0]):  # NEW
        for cls_name in ARTIFACT_TARGETS:  # NEW
            cls_col = ARTIFACT_CLASSES_IDX.get(cls_name)  # NEW
            if cls_col is not None and proba_matrix[comp_idx, cls_col] >= PROB_THRESHOLD:  # NEW
                artifact_idx_prob.append(comp_idx)  # NEW
                break  # NEW: one match per component is enough

    # eog_indices, eog_scores = None, None  (did not exist in the previous version of this file)
    # FIX (Problem: no direct EOG-correlation check): code 1 (the reference implementation) relies
    # primarily on mne.preprocessing.ICA.find_bads_eog(), which correlates each ICA component's time
    # course directly against the RECORDED VEOG channel -- a ground-truth signal, independent of
    # ICLabel's scalp-topography/spectrum-based classifier. Code 2 previously relied ONLY on
    # ICLabel (via argmax and/or probability threshold), which never consults the VEOG/HEOG channels
    # that are already loaded and typed as 'eog' in load_eeg_data(). Adding find_bads_eog() here
    # gives ICA exclusion a second, independent detection path so that a genuine eye-blink component
    # that ICLabel under-scores (e.g. due to the rank/dilution issue fixed above) can still be
    # caught via its correlation with the real VEOG signal, matching code 1's methodology.
    eog_indices_veog, eog_scores_veog = [], None  # NEW: defaults in case VEOG channel is missing
    eog_indices_heog, eog_scores_heog = [], None  # NEW: defaults in case HEOG channel is missing
    try:  # NEW
        if 'VEOG' in raw.ch_names:  # NEW: raw is the ORIGINAL (unfiltered) signal passed into this function, matching code 1's use of raw_input
            eog_indices_veog, eog_scores_veog = ica.find_bads_eog(raw, ch_name='VEOG', threshold=2.0, verbose=False)  # NEW
            print(f"  find_bads_eog(VEOG) flagged components: {eog_indices_veog}")  # NEW
        if 'HEOG' in raw.ch_names:  # NEW: also check HEOG (horizontal eye movements), which code 1 did not check but is available in this dataset
            eog_indices_heog, eog_scores_heog = ica.find_bads_eog(raw, ch_name='HEOG', threshold=2.0, verbose=False)  # NEW
            print(f"  find_bads_eog(HEOG) flagged components: {eog_indices_heog}")  # NEW
    except Exception as e:  # NEW
        print(f"  find_bads_eog failed: {e}")  # NEW
#########

    
    # NEW (methodological decision, not a named standard algorithm -- see justification below):
    # Gate EOG-correlation-only hits (components flagged ONLY via find_bads_eog, with no ICLabel
    # support) behind an ICLabel 'brain' probability ceiling. Rationale: EOG-correlation is a
    # well-established artifact-detection method (Jung et al. 1998/2000; implemented in EEGLAB,
    # FieldTrip, MNE), and combining multiple independent artifact detectors is itself a recognized
    # practice (e.g. SASICA, Chaumon et al. 2015, J. Neurosci. Methods, which combines EOG-correlation,
    # autocorrelation, topography focus, and discontinuity as independent criteria). However, the
    # literature also documents that EOG-correlation alone can produce false positives, since some
    # genuine brain sources (e.g. frontal activity related to eye-movement planning / corollary
    # discharge) can correlate with EOG without being ocular artifacts themselves. The specific
    # threshold below (BRAIN_PROB_CEILING = 0.5) is NOT a published/validated cutoff from any single
    # paper -- it is a conservative engineering choice made for this thesis to reduce the risk of
    # discarding real brain signal when a component's only supporting evidence is EOG-correlation.
    # Any component that ALSO has independent ICLabel support (argmax or probability threshold) is
    # NOT affected by this gate and is still excluded regardless of its 'brain' probability.
    BRAIN_PROB_CEILING = 0.5  # NEW: conservative, thesis-specific choice -- not a literature-standard value
    BRAIN_CLASS_COL = ARTIFACT_CLASSES_IDX['brain']  # NEW
    
    eog_only_idx = (set(eog_indices_veog) | set(eog_indices_heog)) - (set(artifact_idx_argmax) | set(artifact_idx_prob))  # NEW: components flagged ONLY by EOG-correlation, with no ICLabel support at all
    eog_only_rescued = {idx for idx in eog_only_idx if proba_matrix[idx, BRAIN_CLASS_COL] >= BRAIN_PROB_CEILING}  # NEW: rescue if ICLabel itself thinks it's likely brain signal
    if eog_only_rescued:  # NEW
        print(f"  Rescued from EOG-only rejection (ICLabel brain-prob >= {BRAIN_PROB_CEILING}): {sorted(eog_only_rescued)}")  # NEW
    
    artifact_idx = sorted(
        (set(artifact_idx_argmax) | set(artifact_idx_prob) | set(eog_indices_veog) | set(eog_indices_heog))
        - eog_only_rescued
    )  # CHANGED: same union as before, minus the conservative brain-probability rescue applied only to EOG-only hits


    
###########
    # NEW: detailed log for each flagged component
    all_class_names = ['brain', 'muscle artifact', 'eye blink', 'heart beat', 'line noise', 'channel noise', 'other']  # NEW
    for comp_idx in artifact_idx:  # NEW
        probs_str = ", ".join([f"{all_class_names[c]}={proba_matrix[comp_idx, c]:.2f}"  # NEW
                               for c in range(7)])  # NEW
        reason = []  # NEW
        if comp_idx in artifact_idx_argmax:  # NEW
            reason.append(f"argmax='{ic_labels['labels'][comp_idx]}'")  # NEW
        if comp_idx in artifact_idx_prob:  # NEW
            reason.append(f"prob>={PROB_THRESHOLD}")  # NEW
        if comp_idx in eog_indices_veog:  # NEW
            reason.append("VEOG-correlation")  # NEW
        if comp_idx in eog_indices_heog:  # NEW
            reason.append("HEOG-correlation")  # NEW
        print(f"  IC{comp_idx} flagged [{', '.join(reason)}]: {probs_str}")  # NEW

    ica.exclude = artifact_idx
    if len(ica.exclude) > 0:
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(ica.exclude)} ICA components")
    else:
        print("Removed 0 ICA components (no artifact ICs detected)")  # NEW
    

####
        
    
    # Case-insensitive channel drop
    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ['VEOG', 'HEOG', 'TP9']]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")

    # NEW (user request, updated): save before/after preprocessing QC plots for EEG showing ALL
    # channels at once (not just one representative channel). Two SEPARATE figures are produced:
    # one figure with all RAW channels stacked, one figure with all PREPROCESSED channels stacked.
    # Each channel is plotted on its own horizontal row using a fixed vertical offset (the standard
    # "butterfly/stacked montage" view used by EEG viewers such as MNE's raw.plot()), so all channels
    # remain visually separable in a single image instead of being overlaid on top of each other.
    try:  # NEW
        def _plot_all_channels_stacked(raw_obj, title, color, tmin=None, tmax=None):  # CHANGED: added tmin/tmax for zoomed view
            sfreq_plot = raw_obj.info['sfreq']               # NEW
            if tmin is not None or tmax is not None:         # NEW: optional time-window crop
                start_sample = int((tmin or 0.0) * sfreq_plot)             # NEW
                stop_sample = int((tmax * sfreq_plot)) if tmax is not None else None  # NEW
                data = raw_obj.get_data(start=start_sample, stop=stop_sample)         # NEW
                t = (np.arange(data.shape[1]) + start_sample) / sfreq_plot            # NEW
            else:                                            # NEW
                data = raw_obj.get_data()                      # NEW: (n_channels, n_times)
                t = np.arange(data.shape[1]) / sfreq_plot        # NEW
            ch_names_plot = raw_obj.ch_names                # NEW

            # Use the robust (median absolute deviation based) per-channel spread to set a fixed
            # vertical offset between channels, so that channels with very different amplitudes
            # (e.g. after CAR/ICA vs raw) are still readable and non-overlapping within the same figure.
            #spread = np.median(np.abs(data - np.median(data, axis=1, keepdims=True))) * 8 + 1e-12  # NEW
            spread = np.median(np.abs(data - np.median(data, axis=1, keepdims=True))) * 30 + 1e-12  # NEW
            n_ch_plot = data.shape[0]                        # NEW
            #fig_h = max(6, 0.28 * n_ch_plot)  # NEW: scale figure height with channel count
            fig_h = max(6, 0.5 * n_ch_plot)                 # NEW: scale figure height with channel count
            fig, ax = plt.subplots(figsize=(14, fig_h))      # NEW
            for i in range(n_ch_plot):                       # NEW
                offset = (n_ch_plot - 1 - i) * spread         # NEW: top channel = first channel name
                ax.plot(t, data[i] + offset, color=color, linewidth=0.4)  # NEW
            ax.set_yticks([(n_ch_plot - 1 - i) * spread for i in range(n_ch_plot)])  # NEW
            ax.set_yticklabels(ch_names_plot, fontsize=6)    # NEW
            ax.set_xlabel("Time (s)")                        # NEW
            ax.set_ylabel("Channel")                         # NEW
            ax.set_title(title)                              # NEW
            plt.tight_layout()                                # NEW
            return fig                                        # NEW

        fig_before = _plot_all_channels_stacked(  # NEW
            raw_before_qc, "EEG RAW - All Channels (before preprocessing)", 'tab:red'  # NEW
        )  # NEW
        fig_after = _plot_all_channels_stacked(  # NEW
            raw_filtered, "EEG PREPROCESSED - All Channels (after notch+bandpass+CAR+ICA)", 'tab:blue'  # NEW
        )  # NEW
        fig_before_zoom = _plot_all_channels_stacked(  # NEW: zoomed 10-30s view
            raw_before_qc, "EEG RAW - All Channels (before preprocessing, 10-30s)", 'tab:red', tmin=10.0, tmax=30.0  # NEW
        )  # NEW
        fig_after_zoom = _plot_all_channels_stacked(  # NEW: zoomed 10-30s view
            raw_filtered, "EEG PREPROCESSED - All Channels (after notch+bandpass+CAR+ICA, 10-30s)", 'tab:blue', tmin=10.0, tmax=30.0  # NEW
        )  # NEW
        

        # The subject_id is not directly available inside this function (it only receives raw/events),
        # so the figures are saved using generic names inside OUTPUT_PATH/_tmp_eeg_qc and renamed by
        # the caller (process_subject) right after this function returns. This keeps this function's
        # signature and call sites unchanged.
        tmp_qc_dir = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc")  # NEW
        os.makedirs(tmp_qc_dir, exist_ok=True)  # NEW
        fig_before.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_allchannels.png"), dpi=150)  # NEW
        fig_after.savefig(os.path.join(tmp_qc_dir, "last_eeg_after_allchannels.png"), dpi=150)  # NEW
        fig_before_zoom.savefig(os.path.join(tmp_qc_dir, "last_eeg_before_allchannels_zoom10_30.png"), dpi=150)  # NEW
        fig_after_zoom.savefig(os.path.join(tmp_qc_dir, "last_eeg_after_allchannels_zoom10_30.png"), dpi=150)  # NEW
        plt.close(fig_before)  # NEW
        plt.close(fig_after)  # NEW
        plt.close(fig_before_zoom)  # NEW
        plt.close(fig_after_zoom)  # NEW
    except Exception as e:  # NEW
        print(f"Could not save EEG QC plot: {e}")  # NEW
    
    return raw_filtered


# SYNCHRONIZED SLIDING WINDOW EXTRACTION 
# def extract_synchronized_windows(eeg_raw, fnirs_data, events, sfreq_eeg):
# CHANGED: added eeg_spatial_data parameter -- the pre-computed, spatially-mapped (16x16 grid)
# continuous EEG signal for this subject (shape (n_times, 16, 16), produced once in
# process_subject() via spatial_mapping(), mirroring how fNIRS is spatially mapped once before
# windowing rather than per-window). eeg_raw (the MNE Raw object, channel x time) is still passed
# in and still used, but now ONLY for the amplitude-based artifact rejection check below, since
# that threshold (120 uV) is calibrated in electrode voltage units and is not a meaningful check
# on interpolated grid values.
# def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg):
# CHANGED (CRITICAL SYNCHRONIZATION FIX, see the comment above load_and_preprocess_fnirs()'s return
# statement for the full diagnostic rationale): added fnirs_events_time / fnirs_events_code so each
# EEG trial can be paired with fNIRS's OWN trial marker instead of a position derived from EEG's
# clock, which diagnostic testing showed drifts by ~46-93s across the dataset's 3 class sessions.
def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg,  # CHANGED
                                  fnirs_events_time, fnirs_events_code):  # CHANGED

    print("Extracting synchronized asymmetric sliding windows...")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    
    # support both marker formats 
    # label_map = {
    #     1: 0, 2: 1, 3: 1, 4: 2, 5: 2,
    #     16: 0, 48: 1, 64: 1, 80: 2, 96: 2
    # }
    #
    # valid_codes = [1, 2, 3, 4, 5, 16, 48, 64, 80, 96]
    # target_events = events[np.isin(events[:, 2], valid_codes)]
    # FIX (Problem 2): the codes above (1-5, 16/48/64/80/96) are STIMULUS-level markers that fire
    # roughly every 2 s within a 40 s trial (per Shin et al. 2018 / Dataset_description_MATLAB.pdf,
    # codes 16/48/64/80/96 correspond to individual target/non-target number presentations, not to
    # the start of a cognitive-load trial). Sliding a window from each of these stimulus markers
    # caused massive, redundant overlap between windows anchored to different stimuli within the same
    # trial, rather than a clean traversal of the full 40 s task period for that trial. The correct
    # trial-onset codes, confirmed directly from the VP001 .mat structure (inspect_data.py output:
    # mrk_nback.event.desc contains 144 etc., and the dataset documentation confirms 112/128/144 =
    # 0-/2-/3-back SESSION start), are used instead: each trial-onset marker anchors exactly one
    # 40 s task period, which is then traversed by the (unchanged) sliding-window loop below.
    label_map = EEG_TRIAL_LABEL_MAP  # NEW: trial-onset codes (112/128/144) -> class label, defined at top of script
    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())  # NEW
    target_events = events[np.isin(events[:, 2], valid_codes)]  # NEW: now selects one event per ~40s trial instead of one event per ~2s stimulus
    
    print(f"found {len(target_events)} trial markers")
    if len(target_events) == 0:
        print(f"No trial markers found. Available markers: {np.unique(events[:, 2])}")
        # FIX (trial_ids support): keep the empty-return tuple consistent with the new 5-value
        # return signature (added trial_ids), so callers unpacking 5 values never get a
        # "not enough values to unpack" error on this early-exit path.
        return np.array([]), np.array([]), np.array([]), np.array([]), np.array([])  # CHANGED: was 3 empty arrays, now 5 (added session_ids, trial_ids placeholders)

    # NEW (CRITICAL SYNCHRONIZATION FIX -- see comment above load_and_preprocess_fnirs()'s return
    # statement for the full diagnostic rationale): build fNIRS's OWN trial-onset time array (in
    # seconds), independently of EEG's clock, from fnirs_events_time/fnirs_events_code (loaded in
    # load_and_preprocess_fnirs()). Diagnostic testing confirmed that, for every subject checked,
    # EEG's trial-class sequence and fNIRS's trial-class sequence match exactly 1:1 in order, so
    # trial i in target_events corresponds to trial i in fnirs_trial_times/fnirs_trial_classes --
    # this is what allows pairing them by simple array position below.
    fnirs_sync_ok = False  # NEW
    fnirs_trial_times = None  # NEW
    if fnirs_events_time is not None and fnirs_events_code is not None:  # NEW
        fnirs_valid_codes = list(FNIRS_TRIAL_LABEL_MAP.keys())  # NEW
        fnirs_mask = np.isin(fnirs_events_code, fnirs_valid_codes)  # NEW
        fnirs_trial_times_ms = np.asarray(fnirs_events_time)[fnirs_mask].astype(float)  # NEW
        fnirs_trial_codes = np.asarray(fnirs_events_code)[fnirs_mask]  # NEW
        fnirs_order = np.argsort(fnirs_trial_times_ms)  # NEW
        fnirs_trial_times = fnirs_trial_times_ms[fnirs_order] / 1000.0  # NEW: seconds, fNIRS's own clock
        fnirs_trial_classes = np.array([FNIRS_TRIAL_LABEL_MAP[c] for c in fnirs_trial_codes[fnirs_order]])  # NEW

        eeg_trial_classes = np.array([label_map.get(int(ev[2]), -1) for ev in target_events])  # NEW
        n_common = min(len(fnirs_trial_classes), len(eeg_trial_classes))  # NEW
        if (len(fnirs_trial_times) == len(target_events)  # NEW: same trial count in both devices
                and np.array_equal(fnirs_trial_classes[:n_common], eeg_trial_classes[:n_common])):  # NEW: same class order
            fnirs_sync_ok = True  # NEW
            print(f"fNIRS-own-marker sync check: OK ({len(fnirs_trial_times)} trials, class order matches EEG)")  # NEW
        else:
            print(f"  WARNING: fNIRS own markers do not match EEG 1:1 "  # NEW
                  f"(fNIRS n={len(fnirs_trial_times)}, EEG n={len(target_events)}) -- "  # NEW
                  f"falling back to EEG-clock-derived fNIRS timing for this subject.")  # NEW
    else:
        print("  WARNING: fNIRS event markers unavailable -- falling back to EEG-clock-derived fNIRS timing.")  # NEW
    
    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    
    max_slide_seconds = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    step_seconds = EEG_STEP
    
    eeg_windows = []
    fnirs_windows = []
    labels = []
    session_ids = []   # CROSS-SESSION: session number (1,2,3) برای هر window
    trial_ids = []     # NEW (trial_ids fix): trial index (0..N-1, per-subject) for each window -- required by
                        # NEW: Training's trial-level 4:1 val split (run_training reads
                        # NEW: {subject}_trial_ids.npy). Previously this array was never computed or saved,
                        # NEW: which made Training crash with FileNotFoundError the moment it ran.
    
    for trial_idx, ev in enumerate(target_events):  # CHANGED: enumerate() added to index into fnirs_trial_times
        event_sample_eeg = ev[0]
        raw_code = ev[2]
        
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        # CROSS-SESSION: session_id از ترتیب زمانی trial استنتاج می‌شود
        # هر subject 27 trial دارد (3 session × 9 trial per session)
        # trial_idx 0..8 → session 1، trial_idx 9..17 → session 2، trial_idx 18..26 → session 3
        TRIALS_PER_SESSION = 9
        session_id = trial_idx // TRIALS_PER_SESSION + 1
        
        # event_time = event_sample_eeg / sfreq_eeg
        # CHANGED (CRITICAL SYNCHRONIZATION FIX): event_time used to be derived purely from EEG's
        # own clock and then reused to index INTO fNIRS's array too, silently assuming a shared
        # t=0 between the two independently-recorded devices. eeg_event_time (below) is still used
        # for the EEG window and for the baseline window (both correctly indexed into EEG-derived
        # arrays / fnirs_data using EEG's own timeline as before -- baseline correction was already
        # working correctly within a session since the drift is BETWEEN sessions, not within one,
        # and baseline windows are always taken from shortly before the SAME trial). fnirs_event_time
        # (used only for the fNIRS window's actual position below) now comes from fNIRS's OWN trial
        # marker when the sync check above passed, eliminating the ~46-93s cross-session drift.
        eeg_event_time = event_sample_eeg / sfreq_eeg  # CHANGED: renamed from event_time for clarity
        if fnirs_sync_ok:  # NEW
            fnirs_event_time = fnirs_trial_times[trial_idx]  # NEW: fNIRS's own clock for this trial
        else:  # NEW
            fnirs_event_time = eeg_event_time  # NEW: fallback -- old (drift-prone) behavior, only used if sync check failed

        # NEW (isolated baseline-correction test): compute this trial's fNIRS baseline ONCE per
        # trial (not per segment/offset), from the continuous fnirs_data signal, using the
        # BASELINE_TMIN/BASELINE_TMAX window (-5 to -2 s before this trial's onset, on the SAME
        # sync-fixed fnirs_event_time anchor used for the segments themselves). Subtracted from
        # every fNIRS segment belonging to this trial below. If the pre-trial baseline window
        # falls before the start of the recording (only possible for a trial very close to time
        # 0), no correction is applied for that trial (fnirs_baseline stays 0) rather than
        # raising an error or using an out-of-bounds/negative index.
        baseline_start_fnirs = int(np.round((fnirs_event_time + BASELINE_TMIN) * SFREQ_FNIRS))  # NEW
        baseline_end_fnirs = int(np.round((fnirs_event_time + BASELINE_TMAX) * SFREQ_FNIRS))    # NEW
        if baseline_start_fnirs >= 0 and baseline_end_fnirs > baseline_start_fnirs:  # NEW
            fnirs_baseline = fnirs_data[baseline_start_fnirs:baseline_end_fnirs].mean(axis=0, keepdims=True)  # NEW: shape (1,16,16,2)
        else:  # NEW
            fnirs_baseline = 0.0  # NEW: not enough pre-trial data (e.g. the very first trial) -- skip correction for this trial

        for offset_sec in np.arange(0.0, max_slide_seconds + 0.001, step_seconds):
            
            # EEG window
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            
            # fNIRS window 
            # w_start_fnirs = int(np.round((event_time + offset_sec) * SFREQ_FNIRS))
            # w_start_fnirs = int(np.round((fnirs_event_time + offset_sec) * SFREQ_FNIRS))  # CHANGED: uses fNIRS's own clock (fnirs_event_time), not EEG-derived event_time
            # w_end_fnirs = w_start_fnirs + window_samples_fnirs
            #
            # if w_end_eeg > eeg_raw.n_times:
            #     continue
            #
            # if w_end_fnirs > fnirs_data.shape[0]:
            #     continue
            # CHANGED (EXPERIMENTAL -- 11-segment fNIRS windowing): instead of ONE continuous
            # 10s fNIRS window anchored at fnirs_event_time+offset_sec, extract FNIRS_NUM_SEGMENTS=11
            # overlapping 3s segments, each shifted by FNIRS_STEP=1s from the previous one (matching
            # the TSFNet paper's approach, and the same segment structure used in the other expert's
            # extract_tsfnet_windows()). The sync-fix anchor (fnirs_event_time, from fNIRS's OWN
            # trial marker -- completely UNCHANGED from above) is still used as the base time for
            # segment 0; this is the one improvement over the other expert's version, which anchored
            # segments to EEG's clock directly and was therefore subject to the ~46-93s cross-session
            # drift this project already diagnosed and fixed.
            if w_end_eeg > eeg_raw.n_times:  # CHANGED: moved this check up, unchanged logic
                continue

            fnirs_segments = []  # NEW
            valid_segments = True  # NEW
            for seg_idx in range(FNIRS_NUM_SEGMENTS):  # NEW
                seg_offset = offset_sec + seg_idx * FNIRS_STEP  # NEW: each segment shifted by 1s from the previous
                w_start_fnirs_seg = int(np.round((fnirs_event_time + seg_offset) * SFREQ_FNIRS))  # NEW: still uses the sync-fixed anchor
                w_end_fnirs_seg = w_start_fnirs_seg + window_samples_fnirs  # NEW: window_samples_fnirs now = 3s worth of samples (FNIRS_WINDOW_LEN=3.0)

                if w_end_fnirs_seg > fnirs_data.shape[0] or w_start_fnirs_seg < 0:  # NEW
                    valid_segments = False  # NEW
                    break  # NEW

                fnirs_segments.append(fnirs_data[w_start_fnirs_seg:w_end_fnirs_seg] - fnirs_baseline)  # CHANGED: subtract this trial's baseline, isolated from the filter change

            if not valid_segments:  # NEW
                continue  # NEW
            
            # Extract EEG
            # eeg_slice = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)
            # if np.max(np.abs(eeg_slice)) > 120e-6:
            #     continue
            # CHANGED (EEG spatial mapping): the SAVED EEG window must now be the spatially-mapped
            # (16, 16, 600) grid representation, not the raw (28, 600) channel-space slice. The
            # artifact-rejection amplitude check, however, stays on the raw channel-space slice
            # (eeg_slice_raw) since the 120 uV threshold is only meaningful in electrode voltage
            # units, not on cubic-interpolated grid values.
            eeg_slice_raw = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)  # CHANGED: renamed, QC-only now
            if np.max(np.abs(eeg_slice_raw)) > 120e-6:  # CHANGED: uses eeg_slice_raw
                continue

            # NEW: slice the pre-computed spatially-mapped EEG (n_times, 16, 16) at the SAME sample
            # range as eeg_slice_raw (both index the same continuous recording at sfreq_eeg=200Hz),
            # then transpose (600, 16, 16) -> (16, 16, 600) i.e. grid-H x grid-W x time, per spec.
            eeg_slice_spatial = eeg_spatial_data[w_start_eeg:w_end_eeg]  # NEW: (600, 16, 16)
            eeg_slice = eeg_slice_spatial.transpose(1, 2, 0)             # NEW: (16, 16, 600) -- this is what gets saved

            # Extract fNIRS
            # fnirs_slice = fnirs_data[w_start_fnirs:w_end_fnirs]
            # fnirs_slice = fnirs_slice.transpose(3, 0, 1, 2) 
            # CHANGED (EXPERIMENTAL -- 11-segment fNIRS windowing): stack the 11 already-extracted
            # segments (each (30,16,16,2)) into (11,30,16,16,2), then transpose to put the segment
            # axis first and channel (HbO/HbR) axis last, matching the shape convention used by the
            # other expert's implementation of this same approach: (11, 16, 16, 30, 2).
            fnirs_slice = np.stack(fnirs_segments, axis=0)      # NEW: (11, 30, 16, 16, 2)
            fnirs_slice = fnirs_slice.transpose(0, 2, 3, 1, 4)  # NEW: (11, 16, 16, 30, 2)
            
            eeg_windows.append(eeg_slice)
            fnirs_windows.append(fnirs_slice)
            labels.append(label)
            session_ids.append(session_id)   # CROSS-SESSION
            trial_ids.append(trial_idx)      # NEW (trial_ids fix): records which trial (0..26) this window came from,
                                              # NEW: needed by Training's trial-level 4:1 split to guarantee that no
                                              # NEW: window from a given trial leaks across train/val.
    
    eeg_windows = np.array(eeg_windows)
    fnirs_windows = np.array(fnirs_windows)
    labels = np.array(labels)
    session_ids = np.array(session_ids, dtype=np.int32)   # CROSS-SESSION
    trial_ids = np.array(trial_ids, dtype=np.int32)       # NEW (trial_ids fix)
    
    print(f"Extracted {len(labels)} synchronized windows")
    if len(labels) > 0:
        print(f"Label distribution: {np.bincount(labels)}")
    
    return eeg_windows, fnirs_windows, labels, session_ids, trial_ids   # CHANGED: trial_ids added (was: ..., session_ids only)



LOG_FILE = os.path.join(OUTPUT_PATH, "processing_log.txt")  # NEW: log file path

def log_message(msg):  # NEW: helper to print + append to log file
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line)
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")



# MAIN PROCESSING FUNCTION
def process_subject(subject_id):
    t0 = time.time()  # NEW
    log_message(f"START processing {subject_id}")  # NEW
    print(f"Processing {subject_id}")
    
    # subject_num = subject_id[2:]
    # subject_path = os.path.join(BASE_DATASET_PATH, subject_num)
    # eeg_folder = os.path.join(subject_path, f"{subject_id}-EEG")
    # FIX: same flat-folder-structure fix as in load_and_preprocess_fnirs() above; EEG data sits
    # directly under BASE_DATASET_PATH\VPxxx-EEG, with no intermediate numeric subfolder.
    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")  # NEW: flat folder structure
    
    if not os.path.exists(eeg_folder):
        log_message(f"SKIPPED {subject_id}: EEG folder not found")  # NEW
        print(f"EEG folder not found: {eeg_folder}")
        return
    
    try:
        print("\n Loading EEG...")
        # raw_eeg, events, sfreq_eeg = load_eeg_data(eeg_folder)
        raw_eeg, events, sfreq_eeg, eeg_mnt_struct = load_eeg_data(eeg_folder)  # CHANGED: also receive the real EEG mnt_nback.mat struct
        
        print("\n Loading and preprocessing fNIRS...")
        # fnirs_data = load_and_preprocess_fnirs(subject_id)
        fnirs_data, fnirs_events_time, fnirs_events_code = load_and_preprocess_fnirs(subject_id)  # CHANGED: also receive fNIRS's own trial markers
        
        # NEW (bug fix): normalization must happen BEFORE window extraction, not after.
        # extract_synchronized_windows() below slices fnirs_windows directly out of fnirs_data,
        # so if fnirs_data is normalized AFTER that call, the windows already saved are un-normalized
        # and the later normalization line has no effect on the saved output (confirmed by
        # VP011's quality report: fNIRS std=0.000367 instead of ~1.0). Moving the normalization
        # here, right after fnirs_data is produced and before any windowing happens, fixes this.
        print("\n Normalizing fNIRS (subject-level, before windowing)...")  # NEW
        fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)  # NEW: moved here from below
        
        print("\n Preprocessing EEG...")
        raw_eeg_clean = preprocess_eeg(raw_eeg, events)

        # NEW (save cleaned continuous EEG for QC): raw_eeg_clean is the final, fully cleaned
        # channel-space signal (notch + 1-40Hz bandpass + bad-channel interpolation + average
        # reference + ICA artifact removal + VEOG/HEOG/TP9 dropped), saved here IMMEDIATELY after
        # preprocess_eeg() returns -- i.e. before spatial mapping / normalization / windowing
        # convert it into the 16x16 grid tensor representation below, at which point it is no
        # longer available in this channel x time form as a regular mne.Raw object. This .fif file
        # is what load_clean_from_pipeline() in eeg_preprocessing_qc.py looks for first, before
        # falling back to a lossy approximation reconstructed from the windowed data.
        sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW: defined early so the .fif can be saved now (re-defined harmlessly later via os.makedirs(exist_ok=True) before the .npy saves)
        os.makedirs(sub_out_dir, exist_ok=True)  # NEW
        raw_eeg_clean.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_clean_raw.fif"), overwrite=True)  # NEW

        # NEW (EEG per-subject normalization fix support): compute per-channel mean/std from the
        # FULL cleaned, continuous EEG signal (after filtering/CAR/ICA) BEFORE any windowing happens.
        # FIX (per-window vs per-subject normalization): the previous EEG normalization (further
        # below, in the "Normalizing..." section) computed a separate z-score for EACH extracted
        # window independently (axis=-1 over the window's 600 time samples). With a window as short
        # as 3 s, this local z-scoring is noisy/unstable and, more importantly, completely erases any
        # systematic difference in absolute EEG amplitude/power between subjects or between cognitive
        # load levels within a subject -- the model can then only see the RELATIVE waveform shape
        # inside each isolated 3 s window, which is a much weaker and more subject-specific signal.
        # This is very likely a major contributor to the severe overfitting observed during model
        # training (train acc ~96% vs val/test acc near chance level): the model latches onto
        # subject-specific residual noise patterns within each locally-renormalized window instead of
        # learning generalizable cognitive-load-related features. Computing mean/std once per subject
        # over the full continuous recording (mirroring the fNIRS per-subject normalization fix above)
        # and applying those FIXED per-channel statistics to every window extracted from that subject
        # preserves the relative amplitude structure across windows and across load levels, which
        # should generalize better to unseen subjects.
        eeg_full_data = raw_eeg_clean.get_data()  # NEW: shape (n_channels, n_times), fully preprocessed continuous signal
        eeg_subject_mean = eeg_full_data.mean(axis=-1, keepdims=True)  # NEW: shape (n_channels, 1)
        eeg_subject_std = eeg_full_data.std(axis=-1, keepdims=True) + 1e-8  # NEW: shape (n_channels, 1)
        print(f"\n EEG per-subject normalization stats computed: "  # NEW
              f"mean range [{eeg_subject_mean.min():.2e}, {eeg_subject_mean.max():.2e}], "  # NEW
              f"std range [{eeg_subject_std.min():.2e}, {eeg_subject_std.max():.2e}]")  # NEW

        # NEW (EEG spatial mapping, TSFNet paper Sec. 2.3): project the EEG electrodes onto a 16x16
        # spatial grid via azimuthal equidistant projection, exactly as already done for fNIRS
        # (spatial_mapping() + mnt), instead of leaving EEG as raw (channel x time). This is applied
        # ONCE to the full continuous, already-cleaned recording (not per-window) for the same
        # efficiency reason fNIRS is spatially mapped once in load_and_preprocess_fnirs() before
        # windowing: with 66% window overlap, per-window mapping would re-run cubic griddata
        # interpolation on largely the same timepoints hundreds of times over.
        print("\n Spatial mapping EEG to 16x16...")  # NEW
        # eeg_mnt = get_eeg_mnt(raw_eeg_clean)  # electrode positions in raw_eeg_clean.ch_names order (post channel-drop)
        eeg_mnt = get_eeg_mnt(raw_eeg_clean, eeg_mnt_struct)  # CHANGED: uses REAL mnt_nback.mat positions, matched by clab name, not MNE's guessed standard_1005
        eeg_spatial_full = spatial_mapping(eeg_full_data.T, eeg_mnt)  # NEW: (n_channels,n_times).T -> (n_times, n_channels) in, (n_times, 16, 16) out
        print(f"EEG spatially mapped: {eeg_spatial_full.shape}")  # NEW

        # NEW: per-subject normalization of the spatially-mapped EEG, applied to the FULL continuous
        # grid signal before windowing -- mirrors exactly how fNIRS is normalized in this same
        # function (fnirs_data = (fnirs_data - mean)/std over axis=0, i.e. per grid-cell across
        # time). This supersedes the old per-channel normalization (eeg_subject_mean/eeg_subject_std
        # above), which no longer applies now that EEG windows are (16,16,600) grid data rather than
        # (28,600) channel data; eeg_subject_mean/eeg_subject_std are kept above only for the
        # informational print (channel-space signal scale), not used for normalization below anymore.
        eeg_spatial_full = (eeg_spatial_full - eeg_spatial_full.mean(axis=0, keepdims=True)) / (  # NEW
            eeg_spatial_full.std(axis=0, keepdims=True) + 1e-8)  # NEW

        # NEW (user request): finalize the EEG before/after QC plots (now two all-channel figures)
        # for this specific subject by renaming the temp files produced inside preprocess_eeg()
        # (which has no access to subject_id).
        try:  # NEW
            tmp_qc_before = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_allchannels.png")  # NEW
            tmp_qc_after = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_after_allchannels.png")    # NEW
            tmp_qc_before_zoom = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_before_allchannels_zoom10_30.png")  # NEW
            tmp_qc_after_zoom = os.path.join(OUTPUT_PATH, "_tmp_eeg_qc", "last_eeg_after_allchannels_zoom10_30.png")    # NEW
            if os.path.exists(tmp_qc_before) and os.path.exists(tmp_qc_after):  # NEW
                final_qc_dir = os.path.join(OUTPUT_PATH, subject_id)  # NEW
                os.makedirs(final_qc_dir, exist_ok=True)  # NEW
                final_qc_before = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_allchannels.png")  # NEW
                final_qc_after = os.path.join(final_qc_dir, f"{subject_id}_eeg_after_allchannels.png")    # NEW
                os.replace(tmp_qc_before, final_qc_before)  # NEW
                os.replace(tmp_qc_after, final_qc_after)    # NEW
                if os.path.exists(tmp_qc_before_zoom) and os.path.exists(tmp_qc_after_zoom):  # NEW
                    final_qc_before_zoom = os.path.join(final_qc_dir, f"{subject_id}_eeg_before_allchannels_zoom10_30.png")  # NEW
                    final_qc_after_zoom = os.path.join(final_qc_dir, f"{subject_id}_eeg_after_allchannels_zoom10_30.png")    # NEW
                    os.replace(tmp_qc_before_zoom, final_qc_before_zoom)  # NEW
                    os.replace(tmp_qc_after_zoom, final_qc_after_zoom)    # NEW
                print(f"Saved EEG before/after (all-channels) QC plots for {subject_id}")  # NEW
        except Exception as e:  # NEW
            print(f"Could not finalize EEG QC plot for {subject_id}: {e}")  # NEW
        
        print("\n Extracting synchronized asymmetric windows...")
        # eeg_windows, fnirs_windows, labels = extract_synchronized_windows(
        #     raw_eeg_clean, fnirs_data, events, sfreq_eeg
        # )
        # CHANGED: pass eeg_spatial_full (the pre-computed 16x16-grid-mapped continuous EEG) in
        # addition to raw_eeg_clean (still needed inside for the channel-space QC amplitude check).
        # eeg_windows, fnirs_windows, labels = extract_synchronized_windows(
        #     raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg
        # )
        # CHANGED (CRITICAL SYNCHRONIZATION FIX): also pass fnirs_events_time/fnirs_events_code so
        # fNIRS windows are anchored to fNIRS's OWN trial markers, not EEG-clock-derived timing.
        # CHANGED (trial_ids fix): also receive trial_ids, now returned by extract_synchronized_windows.
        eeg_windows, fnirs_windows, labels, session_ids, trial_ids = extract_synchronized_windows(   # CHANGED: trial_ids added
            raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg,
            fnirs_events_time, fnirs_events_code
        )
        
        if len(labels) == 0:
            log_message(f"WARNING {subject_id}: no windows extracted")  # NEW
            print(f"No windows extracted for {subject_id}")
            return
        
        print("\n Normalizing...")
        # EEG: (n_windows, n_channels, n_times) 
        # eeg_windows = (eeg_windows - eeg_windows.mean(axis=-1, keepdims=True)) / (eeg_windows.std(axis=-1, keepdims=True) + 1e-8)
        # FIX (per-window -> per-subject EEG normalization): the line above computed a fresh z-score
        # for every single window independently (per-window mean/std over its own 600 time samples).
        # As explained in the comment above where eeg_subject_mean/eeg_subject_std are computed, this
        # erases systematic amplitude differences between cognitive-load levels and between subjects,
        # which is very likely contributing to the poor generalization (near-chance val/test accuracy)
        # observed during model training. Applying the FIXED per-subject statistics computed once
        # from the full continuous recording (broadcast across all windows and all time samples via
        # reshape to (1, n_channels, 1)) fixes this while still correcting for each subject's overall
        # signal scale, mirroring the per-subject normalization already used for fNIRS.
        # eeg_mean_broadcast = eeg_subject_mean.reshape(1, -1, 1)  # (1, n_channels, 1) for broadcasting over (N, n_channels, n_times)
        # eeg_std_broadcast = eeg_subject_std.reshape(1, -1, 1)
        # eeg_windows = (eeg_windows - eeg_mean_broadcast) / eeg_std_broadcast
        # CHANGED (EEG spatial mapping): eeg_windows is now (n_windows, 16, 16, 600) grid data, not
        # (n_windows, 28, 600) channel data, so the per-channel normalization above no longer applies
        # and is removed. EEG normalization now happens EARLIER, on the full continuous spatially-
        # mapped signal (eeg_spatial_full) before windowing -- see the "NEW: per-subject
        # normalization of the spatially-mapped EEG" block above, right after spatial_mapping() is
        # called -- exactly mirroring how fNIRS is normalized in grid space before windowing. No
        # further EEG normalization is needed here; eeg_windows already carries normalized values.
        
        # fNIRS: (n_windows, 2, 100, 16, 16)
        # fnirs_windows = (fnirs_windows - fnirs_windows.mean(axis=2, keepdims=True)) / (fnirs_windows.std(axis=2, keepdims=True) + 1e-8)
        # FIX (Problem 3): ...
        # fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)
        # BUG FIX (moved): this line was dead code -- fnirs_windows had already been sliced from the
        # un-normalized fnirs_data by extract_synchronized_windows() before this point ran. The actual
        # normalization now happens earlier, right after load_and_preprocess_fnirs() returns and
        # before extract_synchronized_windows() is called. See that block above.
        
        print("\n Saving...")
        sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)
        os.makedirs(sub_out_dir, exist_ok=True)
        
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)
        # np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)
        # NEW (user request): for the first verification run, only save outputs for the very first
        # subject so correctness can be checked before running the full 26-subject batch. The save
        # calls below are commented out for all subjects except VP001; uncomment the three np.save
        # lines below (and remove/relax this condition) once the pipeline has been verified.
#        #if subject_id == "VP001":  # NEW: save outputs only for the first subject during verification
        np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)    # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)  # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)               # NEW
        np.save(os.path.join(sub_out_dir, f"{subject_id}_session_ids.npy"), session_ids)   # CROSS-SESSION
        np.save(os.path.join(sub_out_dir, f"{subject_id}_trial_ids.npy"), trial_ids)       # NEW (trial_ids fix): required by Training's trial-level 4:1 split
        # else:                                                                                    # NEW
        #     # Saving suppressed for subjects other than VP001 during this verification run.        # NEW
        #     pass                                                                                  # NEW

        # NEW (adopted from reviewing a third-party implementation, see chat history): a small
        # per-subject metadata.json sidecar file, so the exact parameters/shapes a given
        # preprocessing run used can be checked at a glance without loading the (large) .npy
        # arrays -- useful given this project has gone through many preprocessing variants
        # (Run A-H) and it has repeatedly been unclear afterward exactly which run produced a
        # given set of output files.
        metadata = {  # NEW
            'subject_id': subject_id,  # NEW
            'eeg_shape': list(eeg_windows.shape),  # NEW
            'fnirs_shape': list(fnirs_windows.shape),  # NEW
            'labels_distribution': np.bincount(labels).tolist(),  # NEW
            'eeg_window_len': EEG_WINDOW_LEN,  # NEW
            'eeg_step': EEG_STEP,  # NEW
            'eeg_epoch_tmax': EEG_EPOCH_TMAX,  # NEW: important given this exact parameter caused a major bug in the reviewed third-party code
            'fnirs_num_segments': FNIRS_NUM_SEGMENTS,  # NEW
            'fnirs_window_len': FNIRS_WINDOW_LEN,  # NEW
            'fnirs_step': FNIRS_STEP,  # NEW
            'sfreq_eeg': SFREQ_EEG,  # NEW
            'sfreq_fnirs': SFREQ_FNIRS,  # NEW
            'beer_lambert_applied': False,  # NEW: this pipeline uses oxy_raw/deoxy_raw directly (Run F fix), not calculate_hbo_hbr()
            'fnirs_sync_fix_applied': True,  # NEW: fNIRS windows anchored to fNIRS's own trial markers, not EEG's clock
            'normalization': 'per-subject z-score (EEG grid + fNIRS grid, before windowing)',  # NEW
            'trial_ids_saved': True,  # NEW (trial_ids fix): flags that this run produced {subject_id}_trial_ids.npy
        }
        with open(os.path.join(sub_out_dir, f"{subject_id}_metadata.json"), 'w') as f:  # NEW
            json.dump(metadata, f, indent=2)  # NEW
        
        print(f"\n {subject_id} completed!")
        print(f"EEG windows: {eeg_windows.shape}")
        print(f"fNIRS windows: {fnirs_windows.shape}")
        print(f"Labels: {np.bincount(labels)}")
        elapsed = (time.time() - t0) / 60  # NEW
        log_message(f"DONE {subject_id} in {elapsed:.1f} min | EEG: {eeg_windows.shape} | fNIRS: {fnirs_windows.shape} | Labels: {np.bincount(labels)}")  # NEW
        
    except Exception as e:
        elapsed = (time.time() - t0) / 60  # NEW
        log_message(f"ERROR processing {subject_id} after {elapsed:.1f} min: {str(e)}")  # NEW
        print(f"Error processing {subject_id}: {str(e)}")
        import traceback
        traceback.print_exc()

# RUN
if __name__ == "__main__":
    total_t0 = time.time()  # NEW
    log_message("===== PIPELINE STARTED =====")  # NEW
    print("MULTIMODAL PREPROCESSING PIPELINE")
    print(f"EEG window: {EEG_WINDOW_LEN}s, step: {EEG_STEP}s")
    print(f"fNIRS window: {FNIRS_WINDOW_LEN}s, step: {FNIRS_STEP}s")
    print(f"fNIRS epoch: {FNIRS_EPOCH_TMAX}s")
    print(f"fNIRS sampling rate: {SFREQ_FNIRS} Hz")
    
    # for sub_num in range(1, 27):
    # NEW (verification run): limited to VP001 only, per user's request to first check correctness
    # on a single subject before running the full 26-subject batch. Change back to range(1, 27)
    # once the pipeline has been verified.
#    for sub_num in range(1, 2):  # NEW: VP001 only, for verification
    for sub_num in range(18, 27):  # NEW
        subject_id = f"VP{sub_num:03d}"
        process_subject(subject_id)

    
    print("ALL THE SELECTED SUBJECTS PROCESSED")


[2026-07-22 12:23:12] ===== PIPELINE STARTED =====
MULTIMODAL PREPROCESSING PIPELINE
EEG window: 3.0s, step: 1.0s
fNIRS window: 3.0s, step: 1.0s
fNIRS epoch: 18.0s
fNIRS sampling rate: 10.0 Hz
[2026-07-22 12:23:12] START processing VP018
Processing VP018

 Loading EEG...
Loading EEG from: E:\kntu\Thesis\codes\DATA\VP018-EEG\cnt_nback.mat
EEG loaded: 388265 samples, 200.0 Hz, 567 markers
unique markers: [ 16  48  64  80  96 112 128 144]
EEG mnt_nback.mat loaded: pos_3d shape (3, 30)

 Loading and preprocessing fNIRS...
Loading raw fNIRS from: E:\kntu\Thesis\codes\DATA\VP018-NIRS\cnt_nback.mat
raw fNIRS sampling rate: 10.0 Hz
Raw fNIRS: oxy (20752, 36), deoxy (20752, 36)
Using oxy/deoxy fields directly as HbO/HbR (Beer-Lambert already applied by dataset authors; see Shin et al. 2018 Data Descriptor)...
Skipping wavelet/MARA/lowpass filtering (MSTFDN-style: relying on the model's own large-kernel convolutions for implicit smoothing)...
Resampling from 10.0 Hz to 10.0 Hz...
Spatial mapping

In [4]:
"""
FULLY SELF-CONTAINED CELL -- paste this ENTIRE content into ONE new Jupyter cell and run it.
No other cell needs to run first. This includes every function it needs, inline.

WHAT IT DOES: reprocesses ONLY TARGET_SUBJECT (default "VP004") with stricter bad-segment
rejection before ICA, using the EXACT SAME notch/bandpass/bad-channel/reference/spatial-
mapping/windowing logic as your validated preprocessing_v2_fixed.py (single-band, 1-40Hz
bandpass, single spatial_mapping() call -- NOT the older 5-band decomposition variant), and
overwrites ONLY that subject's output files. It never touches any other subject's data or files.

CHANGED (aligned with preprocessing_v2_fixed.py, see chat history): this version was previously
written against an older 5-band (delta/theta/alpha/beta/gamma) preprocessing variant that used a
0.5-40Hz bandpass and spatial_mapping_multiband() (EEG windows shaped (16,16,5,600)). That no
longer matches your validated single-band pipeline (1.0-40Hz bandpass, plain spatial_mapping(),
EEG windows shaped (16,16,600)). All band-decomposition code has been removed and every filter/
window/shape parameter below now matches preprocessing_v2_fixed.py exactly, so the .npy files
this script overwrites stay fully compatible with the rest of your pipeline (MMapHybridDataset's
memmap shape (N,16,16,600), etc.).

WHAT'S DIFFERENT FROM THE NORMAL PIPELINE: before fitting ICA, segments where any channel
exceeds 100 uV peak-to-peak (stricter than the normal 120 uV check) are annotated as
"BAD_amplitude" and excluded from the ICA fit (reject_by_annotation=True), so large motion
artifacts don't distort the ICA decomposition. This is the ONLY intentional difference from
preprocess_eeg() in preprocessing_v2_fixed.py.

After running, verify with:  run_qc_batch(subjects=[TARGET_SUBJECT], out=r"...")
"""

import os
import numpy as np
import scipy.io as sio
from scipy.signal import resample
from scipy.interpolate import CloughTocher2DInterpolator
from scipy.spatial import Delaunay
import mne
import mne_icalabel

# ============================================================================
# EDIT THESE TO MATCH YOUR MAIN PREPROCESSING SCRIPT EXACTLY
# ============================================================================
TARGET_SUBJECT = "VP004"
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
OUTPUT_PATH       = r"F:\preprocessing-history-thesis\test0.0"
STRICT_AMPLITUDE_UV = 100.0   # tighter than the pipeline's normal 120uV gross-artifact threshold
BAD_SEGMENT_WINDOW_S = 1.0
BAD_SEGMENT_PADDING_S = 0.25
# ============================================================================

# CHANGED: every constant below now matches preprocessing_v2_fixed.py exactly (values, not just names)
SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0
EEG_WINDOW_LEN = 3.0
EEG_STEP = 1.0
EEG_EPOCH_TMAX = 40.0
FNIRS_WINDOW_LEN = 3.0
FNIRS_NUM_SEGMENTS = 11
FNIRS_STEP = 1.0
BASELINE_TMIN = -5.0
BASELINE_TMAX = -2.0
EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}
FNIRS_TRIAL_LABEL_MAP = {7: 0, 8: 1, 9: 2}
# REMOVED: EEG_FREQ_BANDS / EEG_BAND_ORDER -- the validated pipeline (preprocessing_v2_fixed.py)
# is SINGLE-BAND (one 1-40Hz bandpass, one spatial_mapping() call producing (16,16,600) windows),
# not the older 5-band delta/theta/alpha/beta/gamma decomposition this script previously used.


# ----------------------------------------------------------------------------
# LOAD FUNCTIONS (copied from your validated preprocessing_v2_fixed.py, unchanged)
# ----------------------------------------------------------------------------

def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt_struct = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith('__')][0]]
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    if sfreq != SFREQ_EEG:
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')

    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    mrk_struct = mat_mrk[[k for k in mat_mrk.keys() if not k.startswith('__')][0]]
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("Could not find 'time' or 'pos' in marker struct")
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    else:
        event_codes = event_positions.copy()
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes

    mnt_path = os.path.join(eeg_folder, 'mnt_nback.mat')
    mat_eeg_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    eeg_mnt_struct = mat_eeg_mnt[[k for k in mat_eeg_mnt.keys() if not k.startswith('__')][0]]
    return raw, events, sfreq, eeg_mnt_struct


def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)
    r_safe = np.where(r > 0, r, 1.0)
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))
    phi = np.arctan2(y3d, x3d)
    x_coords = theta * np.cos(phi)
    y_coords = theta * np.sin(phi)
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    tri = Delaunay(points)
    values = data.T
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)
    result = interp((grid_x, grid_y))
    return np.transpose(result, (2, 0, 1)).astype(np.float32)
# REMOVED: spatial_mapping_multiband() -- not used by the single-band pipeline; the single
# spatial_mapping() function above (identical to preprocessing_v2_fixed.py) is used instead,
# applied once to the full continuous cleaned signal (see reprocessing section at the bottom).


class _EEGMontageAdapter:
    def __init__(self, pos_3d):
        self.pos_3d = pos_3d


def get_eeg_mnt(raw, eeg_mnt_struct):
    full_clab = [str(c).strip() for c in np.asarray(eeg_mnt_struct.clab).flatten()]
    full_pos_3d = np.asarray(eeg_mnt_struct.pos_3d, dtype=np.float64)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}
    pos_list = []
    for ch in raw.ch_names:
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))
        pos_list.append([0.0, 0.0, 0.0] if idx is None else full_pos_3d[:, idx])
    return _EEGMontageAdapter(np.array(pos_list, dtype=np.float64).T)


def load_and_preprocess_fnirs(subject_id):
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(fnirs_folder, 'mrk_nback.mat')
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fNIRS file not found: {cnt_path}")

    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith('__')][0]]
    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    mnt = mat_mnt[[k for k in mat_mnt.keys() if not k.startswith('__')][0]]

    fnirs_events_time, fnirs_events_code = None, None
    if os.path.exists(mrk_path):
        mat_fmrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
        fmrk = mat_fmrk[[k for k in mat_fmrk.keys() if not k.startswith('__')][0]]
        if hasattr(fmrk, 'time') and hasattr(fmrk, 'event') and hasattr(fmrk.event, 'desc'):
            fnirs_events_time = np.asarray(fmrk.time).flatten()
            fnirs_events_code = np.asarray(fmrk.event.desc).flatten()

    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0
    hbo_raw = np.asarray(cnt.oxy.x, dtype=np.float64)
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)
    hbo_filtered, hbr_filtered = hbo_raw, deoxy_raw  # Beer-Lambert already applied; no extra filtering (MSTFDN-style)

    n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
    hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
    hbr_resampled = resample(hbr_filtered, n_samples, axis=0)

    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)

    return fnirs_data, fnirs_events_time, fnirs_events_code
# REMOVED: apply_band_decomposition() -- not used; the single-band pipeline filters once
# (inside preprocess_eeg_strict below) instead of producing 5 separate band-filtered copies.


def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg,
                                  fnirs_events_time, fnirs_events_code):
    label_map = EEG_TRIAL_LABEL_MAP
    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())
    target_events = events[np.isin(events[:, 2], valid_codes)]
    if len(target_events) == 0:
        return np.array([]), np.array([]), np.array([]), np.array([]), np.array([])

    fnirs_sync_ok, fnirs_trial_times = False, None
    if fnirs_events_time is not None and fnirs_events_code is not None:
        fnirs_valid_codes = list(FNIRS_TRIAL_LABEL_MAP.keys())
        fnirs_mask = np.isin(fnirs_events_code, fnirs_valid_codes)
        fnirs_trial_times_ms = np.asarray(fnirs_events_time)[fnirs_mask].astype(float)
        fnirs_trial_codes = np.asarray(fnirs_events_code)[fnirs_mask]
        fnirs_order = np.argsort(fnirs_trial_times_ms)
        fnirs_trial_times = fnirs_trial_times_ms[fnirs_order] / 1000.0
        fnirs_trial_classes = np.array([FNIRS_TRIAL_LABEL_MAP[c] for c in fnirs_trial_codes[fnirs_order]])
        eeg_trial_classes = np.array([label_map.get(int(ev[2]), -1) for ev in target_events])
        n_common = min(len(fnirs_trial_classes), len(eeg_trial_classes))
        if (len(fnirs_trial_times) == len(target_events)
                and np.array_equal(fnirs_trial_classes[:n_common], eeg_trial_classes[:n_common])):
            fnirs_sync_ok = True

    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    max_slide_seconds = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    step_seconds = EEG_STEP

    eeg_windows, fnirs_windows, labels, session_ids, trial_ids = [], [], [], [], []
    for trial_idx, ev in enumerate(target_events):
        event_sample_eeg, raw_code = ev[0], ev[2]
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        session_id = trial_idx // 9 + 1
        eeg_event_time = event_sample_eeg / sfreq_eeg
        fnirs_event_time = fnirs_trial_times[trial_idx] if fnirs_sync_ok else eeg_event_time

        baseline_start_fnirs = int(np.round((fnirs_event_time + BASELINE_TMIN) * SFREQ_FNIRS))
        baseline_end_fnirs = int(np.round((fnirs_event_time + BASELINE_TMAX) * SFREQ_FNIRS))
        if baseline_start_fnirs >= 0 and baseline_end_fnirs > baseline_start_fnirs:
            fnirs_baseline = fnirs_data[baseline_start_fnirs:baseline_end_fnirs].mean(axis=0, keepdims=True)
        else:
            fnirs_baseline = 0.0

        for offset_sec in np.arange(0.0, max_slide_seconds + 0.001, step_seconds):
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            if w_end_eeg > eeg_raw.n_times:
                continue

            fnirs_segments, valid_segments = [], True
            for seg_idx in range(FNIRS_NUM_SEGMENTS):
                seg_offset = offset_sec + seg_idx * FNIRS_STEP
                w_start_fnirs_seg = int(np.round((fnirs_event_time + seg_offset) * SFREQ_FNIRS))
                w_end_fnirs_seg = w_start_fnirs_seg + window_samples_fnirs
                if w_end_fnirs_seg > fnirs_data.shape[0] or w_start_fnirs_seg < 0:
                    valid_segments = False
                    break
                fnirs_segments.append(fnirs_data[w_start_fnirs_seg:w_end_fnirs_seg] - fnirs_baseline)
            if not valid_segments:
                continue

            eeg_slice_raw = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)
            if np.max(np.abs(eeg_slice_raw)) > 120e-6:
                continue

            # CHANGED (single-band alignment): eeg_spatial_data is now (n_times, 16, 16) -- the
            # plain single-band spatial_mapping() output, exactly as in preprocessing_v2_fixed.py --
            # not the old (n_times, 16, 16, 5) multi-band tensor. The slice is therefore
            # transposed (1, 2, 0) -> (16, 16, 600), matching preprocessing_v2_fixed.py exactly,
            # not the old (1, 2, 3, 0) -> (16, 16, 5, 600) multi-band transpose.
            eeg_slice_spatial = eeg_spatial_data[w_start_eeg:w_end_eeg]  # (600, 16, 16)
            eeg_slice = eeg_slice_spatial.transpose(1, 2, 0)             # CHANGED: (16, 16, 600), was transpose(1,2,3,0)
            fnirs_slice = np.stack(fnirs_segments, axis=0).transpose(0, 2, 3, 1, 4)

            eeg_windows.append(eeg_slice)
            fnirs_windows.append(fnirs_slice)
            labels.append(label)
            session_ids.append(session_id)
            trial_ids.append(trial_idx)

    return (np.array(eeg_windows), np.array(fnirs_windows), np.array(labels),
            np.array(session_ids, dtype=np.int32), np.array(trial_ids, dtype=np.int32))


# ----------------------------------------------------------------------------
# STRICT preprocessing (the actual change vs. your normal pipeline)
# ----------------------------------------------------------------------------

def annotate_bad_segments(raw, threshold_uv=STRICT_AMPLITUDE_UV,
                           window_s=BAD_SEGMENT_WINDOW_S, padding_s=BAD_SEGMENT_PADDING_S):
    data = raw.get_data(picks="eeg")
    if np.median(np.abs(data)) < 1e-3:
        data = data * 1e6
    sfreq = raw.info["sfreq"]
    win = int(window_s * sfreq)
    n_windows = data.shape[1] // win
    bad_mask = np.zeros(n_windows, dtype=bool)
    for w in range(n_windows):
        seg = data[:, w * win:(w + 1) * win]
        if np.ptp(seg, axis=1).max() > threshold_uv:
            bad_mask[w] = True
    onsets, durations, i = [], [], 0
    while i < n_windows:
        if bad_mask[i]:
            start = i
            while i < n_windows and bad_mask[i]:
                i += 1
            onset_s = max(0.0, start * window_s - padding_s)
            end_s = min(raw.times[-1], i * window_s + padding_s)
            onsets.append(onset_s)
            durations.append(end_s - onset_s)
        else:
            i += 1
    print(f"  Flagged {len(onsets)} bad segment(s), {sum(durations):.1f}s total out of "
          f"{raw.times[-1]:.1f}s ({sum(durations)/raw.times[-1]*100:.1f}%)")
    if onsets:
        bad_annot = mne.Annotations(onset=onsets, duration=durations,
                                     description=["BAD_amplitude"] * len(onsets))
        raw.set_annotations(raw.annotations + bad_annot if raw.annotations else bad_annot)
    return raw


def preprocess_eeg_strict(raw, events):
    print("Preprocessing EEG (STRICT bad-segment rejection before ICA)...")
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks="eeg", verbose=False)
    # CHANGED: l_freq 0.5 -> 1.0, matching preprocess_eeg() in preprocessing_v2_fixed.py exactly
    # (this project's validated highpass cutoff is 1.0Hz, not 0.5Hz).
    raw_filtered.filter(l_freq=1.0, h_freq=40.0, method="fir", verbose=False)  # CHANGED: was l_freq=0.5

    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method="fir", verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ["VEOG", "HEOG", "TP9"]:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)
    if bad_channels:
        print(f"Bad channels: {bad_channels}")
        raw_filtered.info["bads"] = bad_channels
    if raw_filtered.info["bads"]:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    raw_filtered.set_eeg_reference("average", projection=False, verbose=False)

    print("  Annotating high-amplitude segments for exclusion from ICA fit...")
    raw_filtered = annotate_bad_segments(raw_filtered)

    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude="bads")
    ica = mne.preprocessing.ICA(n_components=0.99, method="picard",
                                 fit_params=dict(ortho=False, extended=True),
                                 random_state=42, verbose=False)
    ica.fit(raw_filtered, picks=picks_for_ica, reject_by_annotation=True)  # THE ACTUAL CHANGE: strict bad-segment exclusion from ICA fit

    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method="iclabel")
    ARTIFACT_CLASSES_IDX = {"brain": 0, "muscle artifact": 1, "eye blink": 2,
                             "heart beat": 3, "line noise": 4, "channel noise": 5, "other": 6}
    ARTIFACT_TARGETS = ["eye blink", "eye movement", "muscle artifact", "line noise"]
    PROB_THRESHOLD = 0.30
    artifact_idx_argmax = [idx for idx, label in enumerate(ic_labels["labels"]) if label in ARTIFACT_TARGETS]
    artifact_idx_prob = []
    proba_matrix = ica.labels_scores_
    for comp_idx in range(proba_matrix.shape[0]):
        for cls_name in ARTIFACT_TARGETS:
            cls_col = ARTIFACT_CLASSES_IDX.get(cls_name)
            if cls_col is not None and proba_matrix[comp_idx, cls_col] >= PROB_THRESHOLD:
                artifact_idx_prob.append(comp_idx)
                break
    eog_indices_veog, eog_indices_heog = [], []
    try:
        if "VEOG" in raw.ch_names:
            eog_indices_veog, _ = ica.find_bads_eog(raw, ch_name="VEOG", threshold=2.0, verbose=False)
        if "HEOG" in raw.ch_names:
            eog_indices_heog, _ = ica.find_bads_eog(raw, ch_name="HEOG", threshold=2.0, verbose=False)
    except Exception as e:
        print(f"  find_bads_eog failed: {e}")
    BRAIN_PROB_CEILING = 0.5
    BRAIN_CLASS_COL = ARTIFACT_CLASSES_IDX["brain"]
    eog_only_idx = (set(eog_indices_veog) | set(eog_indices_heog)) - (set(artifact_idx_argmax) | set(artifact_idx_prob))
    eog_only_rescued = {idx for idx in eog_only_idx if proba_matrix[idx, BRAIN_CLASS_COL] >= BRAIN_PROB_CEILING}
    artifact_idx = sorted(
        (set(artifact_idx_argmax) | set(artifact_idx_prob) | set(eog_indices_veog) | set(eog_indices_heog))
        - eog_only_rescued
    )
    ica.exclude = artifact_idx
    if len(ica.exclude) > 0:
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(ica.exclude)} ICA components: {artifact_idx}")
    else:
        print("Removed 0 ICA components")

    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ["VEOG", "HEOG", "TP9"]]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")

    return raw_filtered


# ----------------------------------------------------------------------------
# MAIN: reprocess only TARGET_SUBJECT
# ----------------------------------------------------------------------------
print(f"\n{'='*80}\nSTRICT reprocessing: {TARGET_SUBJECT} (does not affect any other subject)\n{'='*80}")

eeg_folder = os.path.join(BASE_DATASET_PATH, f"{TARGET_SUBJECT}-EEG")
raw_eeg, events, sfreq_eeg, eeg_mnt_struct = load_eeg_data(eeg_folder)
fnirs_data, fnirs_events_time, fnirs_events_code = load_and_preprocess_fnirs(TARGET_SUBJECT)
fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)

raw_eeg_clean = preprocess_eeg_strict(raw_eeg, events)

sub_out_dir = os.path.join(OUTPUT_PATH, TARGET_SUBJECT)
os.makedirs(sub_out_dir, exist_ok=True)
raw_eeg_clean.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_eeg_clean_raw.fif"), overwrite=True)
print(f"Saved strict-cleaned continuous EEG to {sub_out_dir}\\{TARGET_SUBJECT}_eeg_clean_raw.fif")

# CHANGED (single-band alignment): the old 5-band decomposition + spatial_mapping_multiband()
# block is replaced with the SAME single-band spatial mapping used in preprocessing_v2_fixed.py --
# one spatial_mapping() call on the full continuous, already-cleaned (1-40Hz, ICA'd) signal,
# followed by the same per-subject z-score normalization (mean/std over the time axis, axis=0).
eeg_mnt = get_eeg_mnt(raw_eeg_clean, eeg_mnt_struct)                       # CHANGED: no band_signals/band_data_list needed
eeg_full_data = raw_eeg_clean.get_data()                                    # (n_channels, n_times)
eeg_spatial_full = spatial_mapping(eeg_full_data.T, eeg_mnt)                # CHANGED: single spatial_mapping() call, matches preprocessing_v2_fixed.py exactly -> (n_times, 16, 16)
eeg_spatial_full = (eeg_spatial_full - eeg_spatial_full.mean(axis=0, keepdims=True)) / (
    eeg_spatial_full.std(axis=0, keepdims=True) + 1e-8)                     # per-subject z-score, same as preprocessing_v2_fixed.py

eeg_windows, fnirs_windows, labels, session_ids, trial_ids = extract_synchronized_windows(
    raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg, fnirs_events_time, fnirs_events_code
)

if len(labels) == 0:
    print(f"No windows extracted for {TARGET_SUBJECT} -- ABORTING save (old files left untouched).")
else:
    np.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_eeg_windows.npy"), eeg_windows)
    np.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_fnirs_windows.npy"), fnirs_windows)
    np.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_labels.npy"), labels)
    np.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_session_ids.npy"), session_ids)
    np.save(os.path.join(sub_out_dir, f"{TARGET_SUBJECT}_trial_ids.npy"), trial_ids)

    print(f"\n{TARGET_SUBJECT} strict-reprocessing complete!")
    print(f"EEG windows: {eeg_windows.shape}")  # CHANGED: now (N, 16, 16, 600), matching preprocessing_v2_fixed.py -- was (N, 16, 16, 5, 600)
    print(f"fNIRS windows: {fnirs_windows.shape}")
    print(f"Labels: {np.bincount(labels)}")
    print(f"\nNext step: re-run QC for {TARGET_SUBJECT} only to confirm the FAIL/WARN items are resolved:")
    print(f'  run_qc_batch(subjects=["{TARGET_SUBJECT}"], out=r"...")')



STRICT reprocessing: VP004 (does not affect any other subject)
Preprocessing EEG (STRICT bad-segment rejection before ICA)...
Bad channels: ['T7', 'CP5', 'P7']
  Annotating high-amplitude segments for exclusion from ICA fit...
  Flagged 424 bad segment(s), 1331.8s total out of 1944.8s (68.5%)
Fitting ICA to data using 28 channels (please be patient, this may take a while)
Omitting 266350 of 388969 (68.48%) samples, retaining 122619 (31.52%) samples.
Selecting by explained variance: 18 components
Fitting ICA took 26.5s.
Removed 2 ICA components: [0, np.int64(7)]
Dropped channels: ['HEOG', 'VEOG']
Overwriting existing file.
Writing F:\preprocessing-history-thesis\test0.0\VP004\VP004_eeg_clean_raw.fif
Overwriting existing file.
Closing F:\preprocessing-history-thesis\test0.0\VP004\VP004_eeg_clean_raw.fif
[done]
Saved strict-cleaned continuous EEG to F:\preprocessing-history-thesis\test0.0\VP004\VP004_eeg_clean_raw.fif

VP004 strict-reprocessing complete!
EEG windows: (712, 16, 16, 600)
f

In [5]:
"""
FULLY SELF-CONTAINED CELL -- paste this ENTIRE content into ONE new Jupyter cell and run it.

WHAT IT DOES: reprocesses VP015, VP018, VP019, VP020 with Fp1 (and Fp2 where flagged) FORCED as
bad channels -- i.e. spherical-spline interpolated from neighbouring electrodes -- BEFORE ICA,
instead of relying on ICA to remove the residual low-frequency contamination there (which the
ICA eye-blink probability deep-dive confirmed does NOT come from a separable ICA source in any
of these 4 subjects). Everything else is identical to your validated preprocessing_v2_fixed.py.

CHANGED (aligned with preprocessing_v2_fixed.py, see chat history): this version was previously
written against an older 5-band (delta/theta/alpha/beta/gamma) preprocessing variant that used a
0.5-40Hz bandpass and spatial_mapping_multiband() (EEG windows shaped (16,16,5,600)). That no
longer matches your validated single-band pipeline (1.0-40Hz bandpass, plain spatial_mapping(),
EEG windows shaped (16,16,600)). All band-decomposition code has been removed and every filter/
window/shape parameter below now matches preprocessing_v2_fixed.py exactly, so the .npy files
this script overwrites stay fully compatible with the rest of your pipeline (MMapHybridDataset's
memmap shape (N,16,16,600), etc.).

Per-subject forced-bad-channel list (from the delta-band cross-check results):
    VP015: Fp1, Fp2
    VP018: Fp1, Fp2
    VP019: Fp1          (Fp2 was NOT flagged for VP019)
    VP020: Fp1, Fp2

This ONLY overwrites these 4 subjects' output files in OUTPUT_PATH. No other subject is touched.

After running, verify with:
    run_qc_batch(subjects=["VP015","VP018","VP019","VP020"], out=r"...")
and/or re-run delta_band_ocular_crosscheck.py restricted to these 4 subjects to confirm the
elevated Fp1/Fp2 delta power is gone.
"""

import os
import numpy as np
import scipy.io as sio
from scipy.signal import resample
from scipy.interpolate import CloughTocher2DInterpolator
from scipy.spatial import Delaunay
import mne
import mne_icalabel

# ============================================================================
# EDIT THESE TO MATCH YOUR MAIN PREPROCESSING SCRIPT EXACTLY
# ============================================================================
BASE_DATASET_PATH = r"E:\kntu\Thesis\codes\DATA"
OUTPUT_PATH       = r"F:\preprocessing-history-thesis\test0.0"

FORCED_BAD_CHANNELS = {
    "VP015": ["Fp1", "Fp2"],
    "VP018": ["Fp1", "Fp2"],
    "VP019": ["Fp1"],
    "VP020": ["Fp1", "Fp2"],
}
# ============================================================================

# CHANGED: every constant below now matches preprocessing_v2_fixed.py exactly (values, not just names)
SFREQ_EEG = 200.0
SFREQ_FNIRS = 10.0
EEG_WINDOW_LEN = 3.0
EEG_STEP = 1.0
EEG_EPOCH_TMAX = 40.0
FNIRS_WINDOW_LEN = 3.0
FNIRS_NUM_SEGMENTS = 11
FNIRS_STEP = 1.0
BASELINE_TMIN = -5.0
BASELINE_TMAX = -2.0
EEG_TRIAL_LABEL_MAP = {112: 0, 128: 1, 144: 2}
FNIRS_TRIAL_LABEL_MAP = {7: 0, 8: 1, 9: 2}
# REMOVED: EEG_FREQ_BANDS / EEG_BAND_ORDER -- the validated pipeline (preprocessing_v2_fixed.py)
# is SINGLE-BAND (one 1-40Hz bandpass, one spatial_mapping() call producing (16,16,600) windows),
# not the older 5-band delta/theta/alpha/beta/gamma decomposition this script previously used.


# ----------------------------------------------------------------------------
# LOAD / SPATIAL FUNCTIONS (identical to your validated preprocessing_v2_fixed.py)
# ----------------------------------------------------------------------------

def load_eeg_data(eeg_folder):
    cnt_path = os.path.join(eeg_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(eeg_folder, 'mrk_nback.mat')
    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt_struct = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith('__')][0]]
    data = cnt_struct.x.T
    sfreq = float(cnt_struct.fs)
    ch_names = [str(ch).strip() for ch in cnt_struct.clab]
    if np.max(np.abs(data)) > 1.0:
        data = data * 1e-6
    if sfreq != SFREQ_EEG:
        n_samples = int(data.shape[1] * SFREQ_EEG / sfreq)
        data = resample(data, n_samples, axis=1)
        sfreq = SFREQ_EEG
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    raw = mne.io.RawArray(data, info, verbose=False)
    eog_channels = [ch for ch in ['VEOG', 'HEOG'] if ch in ch_names]
    if eog_channels:
        raw.set_channel_types({ch: 'eog' for ch in eog_channels})
    raw.set_montage('standard_1005', match_case=False, on_missing='ignore')

    mat_mrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
    mrk_struct = mat_mrk[[k for k in mat_mrk.keys() if not k.startswith('__')][0]]
    if hasattr(mrk_struct, 'time'):
        event_positions = np.round(mrk_struct.time * sfreq / 1000).astype(int)
    elif hasattr(mrk_struct, 'pos'):
        event_positions = np.array(mrk_struct.pos, dtype=int) - 1
    else:
        raise ValueError("Could not find 'time' or 'pos' in marker struct")
    if hasattr(mrk_struct, 'event') and hasattr(mrk_struct.event, 'desc'):
        desc = mrk_struct.event.desc
        if desc.ndim > 1:
            desc = desc.squeeze()
        event_codes = np.asarray(desc, dtype=int).flatten()
    else:
        event_codes = event_positions.copy()
    events = np.zeros((len(event_positions), 3), dtype=int)
    events[:, 0] = event_positions
    events[:, 2] = event_codes

    mnt_path = os.path.join(eeg_folder, 'mnt_nback.mat')
    mat_eeg_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    eeg_mnt_struct = mat_eeg_mnt[[k for k in mat_eeg_mnt.keys() if not k.startswith('__')][0]]
    return raw, events, sfreq, eeg_mnt_struct


def spatial_mapping(data, mnt):
    n_samples, n_channels = data.shape
    pos_3d = np.asarray(mnt.pos_3d, dtype=np.float64)
    x3d, y3d, z3d = pos_3d[0, :], pos_3d[1, :], pos_3d[2, :]
    r = np.sqrt(x3d**2 + y3d**2 + z3d**2)
    r_safe = np.where(r > 0, r, 1.0)
    theta = np.arccos(np.clip(z3d / r_safe, -1.0, 1.0))
    phi = np.arctan2(y3d, x3d)
    x_coords = theta * np.cos(phi)
    y_coords = theta * np.sin(phi)
    grid_x, grid_y = np.mgrid[min(x_coords):max(x_coords):16j, min(y_coords):max(y_coords):16j]
    points = np.column_stack((x_coords, y_coords))
    tri = Delaunay(points)
    values = data.T
    interp = CloughTocher2DInterpolator(tri, values, fill_value=0)
    result = interp((grid_x, grid_y))
    return np.transpose(result, (2, 0, 1)).astype(np.float32)
# REMOVED: spatial_mapping_multiband() -- not used by the single-band pipeline; the single
# spatial_mapping() function above (identical to preprocessing_v2_fixed.py) is used instead,
# applied once to the full continuous cleaned signal (see reprocess_subject_with_forced_bad below).


class _EEGMontageAdapter:
    def __init__(self, pos_3d):
        self.pos_3d = pos_3d


def get_eeg_mnt(raw, eeg_mnt_struct):
    full_clab = [str(c).strip() for c in np.asarray(eeg_mnt_struct.clab).flatten()]
    full_pos_3d = np.asarray(eeg_mnt_struct.pos_3d, dtype=np.float64)
    name_to_idx = {name: i for i, name in enumerate(full_clab)}
    name_to_idx_upper = {name.upper(): i for i, name in enumerate(full_clab)}
    pos_list = []
    for ch in raw.ch_names:
        idx = name_to_idx.get(ch, name_to_idx_upper.get(ch.upper(), None))
        pos_list.append([0.0, 0.0, 0.0] if idx is None else full_pos_3d[:, idx])
    return _EEGMontageAdapter(np.array(pos_list, dtype=np.float64).T)


def load_and_preprocess_fnirs(subject_id):
    fnirs_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-NIRS")
    cnt_path = os.path.join(fnirs_folder, 'cnt_nback.mat')
    mrk_path = os.path.join(fnirs_folder, 'mrk_nback.mat')
    mnt_path = os.path.join(fnirs_folder, 'mnt_nback.mat')
    if not os.path.exists(cnt_path):
        raise FileNotFoundError(f"fNIRS file not found: {cnt_path}")

    mat_cnt = sio.loadmat(cnt_path, struct_as_record=False, squeeze_me=True)
    cnt = mat_cnt[[k for k in mat_cnt.keys() if not k.startswith('__')][0]]
    mat_mnt = sio.loadmat(mnt_path, struct_as_record=False, squeeze_me=True)
    mnt = mat_mnt[[k for k in mat_mnt.keys() if not k.startswith('__')][0]]

    fnirs_events_time, fnirs_events_code = None, None
    if os.path.exists(mrk_path):
        mat_fmrk = sio.loadmat(mrk_path, struct_as_record=False, squeeze_me=True)
        fmrk = mat_fmrk[[k for k in mat_fmrk.keys() if not k.startswith('__')][0]]
        if hasattr(fmrk, 'time') and hasattr(fmrk, 'event') and hasattr(fmrk.event, 'desc'):
            fnirs_events_time = np.asarray(fmrk.time).flatten()
            fnirs_events_code = np.asarray(fmrk.event.desc).flatten()

    sfreq_fnirs_raw = float(cnt.oxy.fs) if hasattr(cnt, 'oxy') and hasattr(cnt.oxy, 'fs') else 10.0
    hbo_raw = np.asarray(cnt.oxy.x, dtype=np.float64)
    deoxy_raw = np.asarray(cnt.deoxy.x, dtype=np.float64)
    hbo_filtered, hbr_filtered = hbo_raw, deoxy_raw  # Beer-Lambert already applied; no extra filtering (MSTFDN-style)

    n_samples = int(hbo_filtered.shape[0] * SFREQ_FNIRS / sfreq_fnirs_raw)
    hbo_resampled = resample(hbo_filtered, n_samples, axis=0)
    hbr_resampled = resample(hbr_filtered, n_samples, axis=0)

    hbo_mapped = spatial_mapping(hbo_resampled, mnt)
    hbr_mapped = spatial_mapping(hbr_resampled, mnt)
    fnirs_data = np.stack([hbo_mapped, hbr_mapped], axis=-1)

    return fnirs_data, fnirs_events_time, fnirs_events_code
# REMOVED: apply_band_decomposition() -- not used; the single-band pipeline filters once
# (inside preprocess_eeg_force_bad below) instead of producing 5 separate band-filtered copies.


def extract_synchronized_windows(eeg_raw, eeg_spatial_data, fnirs_data, events, sfreq_eeg,
                                  fnirs_events_time, fnirs_events_code):
    label_map = EEG_TRIAL_LABEL_MAP
    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())
    target_events = events[np.isin(events[:, 2], valid_codes)]
    if len(target_events) == 0:
        return np.array([]), np.array([]), np.array([]), np.array([]), np.array([])

    fnirs_sync_ok, fnirs_trial_times = False, None
    if fnirs_events_time is not None and fnirs_events_code is not None:
        fnirs_valid_codes = list(FNIRS_TRIAL_LABEL_MAP.keys())
        fnirs_mask = np.isin(fnirs_events_code, fnirs_valid_codes)
        fnirs_trial_times_ms = np.asarray(fnirs_events_time)[fnirs_mask].astype(float)
        fnirs_trial_codes = np.asarray(fnirs_events_code)[fnirs_mask]
        fnirs_order = np.argsort(fnirs_trial_times_ms)
        fnirs_trial_times = fnirs_trial_times_ms[fnirs_order] / 1000.0
        fnirs_trial_classes = np.array([FNIRS_TRIAL_LABEL_MAP[c] for c in fnirs_trial_codes[fnirs_order]])
        eeg_trial_classes = np.array([label_map.get(int(ev[2]), -1) for ev in target_events])
        n_common = min(len(fnirs_trial_classes), len(eeg_trial_classes))
        if (len(fnirs_trial_times) == len(target_events)
                and np.array_equal(fnirs_trial_classes[:n_common], eeg_trial_classes[:n_common])):
            fnirs_sync_ok = True

    window_samples_eeg = int(EEG_WINDOW_LEN * sfreq_eeg)
    window_samples_fnirs = int(FNIRS_WINDOW_LEN * SFREQ_FNIRS)
    max_slide_seconds = EEG_EPOCH_TMAX - EEG_WINDOW_LEN
    step_seconds = EEG_STEP

    eeg_windows, fnirs_windows, labels, session_ids, trial_ids = [], [], [], [], []
    for trial_idx, ev in enumerate(target_events):
        event_sample_eeg, raw_code = ev[0], ev[2]
        label = label_map.get(raw_code, -1)
        if label == -1:
            continue
        session_id = trial_idx // 9 + 1
        eeg_event_time = event_sample_eeg / sfreq_eeg
        fnirs_event_time = fnirs_trial_times[trial_idx] if fnirs_sync_ok else eeg_event_time

        baseline_start_fnirs = int(np.round((fnirs_event_time + BASELINE_TMIN) * SFREQ_FNIRS))
        baseline_end_fnirs = int(np.round((fnirs_event_time + BASELINE_TMAX) * SFREQ_FNIRS))
        if baseline_start_fnirs >= 0 and baseline_end_fnirs > baseline_start_fnirs:
            fnirs_baseline = fnirs_data[baseline_start_fnirs:baseline_end_fnirs].mean(axis=0, keepdims=True)
        else:
            fnirs_baseline = 0.0

        for offset_sec in np.arange(0.0, max_slide_seconds + 0.001, step_seconds):
            w_start_eeg = int(event_sample_eeg + np.round(offset_sec * sfreq_eeg))
            w_end_eeg = w_start_eeg + window_samples_eeg
            if w_end_eeg > eeg_raw.n_times:
                continue

            fnirs_segments, valid_segments = [], True
            for seg_idx in range(FNIRS_NUM_SEGMENTS):
                seg_offset = offset_sec + seg_idx * FNIRS_STEP
                w_start_fnirs_seg = int(np.round((fnirs_event_time + seg_offset) * SFREQ_FNIRS))
                w_end_fnirs_seg = w_start_fnirs_seg + window_samples_fnirs
                if w_end_fnirs_seg > fnirs_data.shape[0] or w_start_fnirs_seg < 0:
                    valid_segments = False
                    break
                fnirs_segments.append(fnirs_data[w_start_fnirs_seg:w_end_fnirs_seg] - fnirs_baseline)
            if not valid_segments:
                continue

            eeg_slice_raw = eeg_raw.get_data(start=w_start_eeg, stop=w_end_eeg)
            if np.max(np.abs(eeg_slice_raw)) > 120e-6:
                continue

            # CHANGED (single-band alignment): eeg_spatial_data is now (n_times, 16, 16) -- the
            # plain single-band spatial_mapping() output, exactly as in preprocessing_v2_fixed.py --
            # not the old (n_times, 16, 16, 5) multi-band tensor. The slice is therefore
            # transposed (1, 2, 0) -> (16, 16, 600), matching preprocessing_v2_fixed.py exactly,
            # not the old (1, 2, 3, 0) -> (16, 16, 5, 600) multi-band transpose.
            eeg_slice_spatial = eeg_spatial_data[w_start_eeg:w_end_eeg]  # (600, 16, 16)
            eeg_slice = eeg_slice_spatial.transpose(1, 2, 0)             # CHANGED: (16, 16, 600), was transpose(1,2,3,0)
            fnirs_slice = np.stack(fnirs_segments, axis=0).transpose(0, 2, 3, 1, 4)

            eeg_windows.append(eeg_slice)
            fnirs_windows.append(fnirs_slice)
            labels.append(label)
            session_ids.append(session_id)
            trial_ids.append(trial_idx)

    return (np.array(eeg_windows), np.array(fnirs_windows), np.array(labels),
            np.array(session_ids, dtype=np.int32), np.array(trial_ids, dtype=np.int32))


# ----------------------------------------------------------------------------
# preprocess_eeg WITH FORCED BAD CHANNELS (the actual change vs. your normal pipeline)
# ----------------------------------------------------------------------------

def preprocess_eeg_force_bad(raw, events, forced_bad_channels):
    print(f"Preprocessing EEG (forcing {forced_bad_channels} as bad -> interpolated before ICA)...")
    raw_filtered = raw.copy().notch_filter(freqs=50.0, picks="eeg", verbose=False)
    # CHANGED: l_freq 0.5 -> 1.0, matching preprocess_eeg() in preprocessing_v2_fixed.py exactly
    # (this project's validated highpass cutoff is 1.0Hz, not 0.5Hz).
    raw_filtered.filter(l_freq=1.0, h_freq=40.0, method="fir", verbose=False)  # CHANGED: was l_freq=0.5

    raw_temp = raw_filtered.copy().filter(l_freq=10.0, h_freq=40.0, method="fir", verbose=False)
    eeg_picks = mne.pick_types(raw_temp.info, eeg=True, eog=False)
    eeg_names = [raw_temp.ch_names[idx] for idx in eeg_picks]
    temp_data = raw_temp.get_data(picks=eeg_picks)
    stds = np.std(temp_data, axis=1)
    median_std = np.median(stds)
    mad = np.median(np.abs(stds - median_std)) if np.median(np.abs(stds - median_std)) > 0 else 1e-6
    corr_matrix = np.corrcoef(temp_data)
    mean_corrs = np.mean(corr_matrix, axis=1)
    bad_channels = []
    for i, ch_name in enumerate(eeg_names):
        if ch_name.upper() in ["VEOG", "HEOG", "TP9"]:
            continue
        if stds[i] > (median_std + 4.5 * mad) or stds[i] < (median_std - 4.5 * mad) or mean_corrs[i] < 0.30:
            bad_channels.append(ch_name)

    # THE ACTUAL CHANGE for this variant: force Fp1/Fp2 (diagnosed root cause -- see chat history)
    # into the bad-channel list in ADDITION to whatever the automatic std/correlation detector
    # found, so they get spherical-spline interpolated from neighbouring electrodes exactly like
    # any other bad channel, BEFORE ICA -- instead of relying on ICA to remove their residual
    # low-frequency contamination, which the ICA eye-blink probability deep-dive confirmed does
    # NOT come from any separable ICA source in these subjects.
    for ch in forced_bad_channels:
        if ch in raw_filtered.ch_names and ch not in bad_channels:
            bad_channels.append(ch)

    if bad_channels:
        print(f"Bad channels (auto-detected + forced): {bad_channels}")
        raw_filtered.info["bads"] = bad_channels
    if raw_filtered.info["bads"]:
        raw_filtered.interpolate_bads(reset_bads=True, verbose=False)
    raw_filtered.set_eeg_reference("average", projection=False, verbose=False)

    picks_for_ica = mne.pick_types(raw_filtered.info, eeg=True, eog=False, exclude="bads")
    ica = mne.preprocessing.ICA(n_components=0.99, method="picard",
                                 fit_params=dict(ortho=False, extended=True),
                                 random_state=42, verbose=False)
    ica.fit(raw_filtered, picks=picks_for_ica)

    ic_labels = mne_icalabel.label_components(raw_filtered, ica, method="iclabel")
    ARTIFACT_CLASSES_IDX = {"brain": 0, "muscle artifact": 1, "eye blink": 2,
                             "heart beat": 3, "line noise": 4, "channel noise": 5, "other": 6}
    ARTIFACT_TARGETS = ["eye blink", "eye movement", "muscle artifact", "line noise"]
    PROB_THRESHOLD = 0.30
    artifact_idx_argmax = [idx for idx, label in enumerate(ic_labels["labels"]) if label in ARTIFACT_TARGETS]
    artifact_idx_prob = []
    proba_matrix = ica.labels_scores_
    for comp_idx in range(proba_matrix.shape[0]):
        for cls_name in ARTIFACT_TARGETS:
            cls_col = ARTIFACT_CLASSES_IDX.get(cls_name)
            if cls_col is not None and proba_matrix[comp_idx, cls_col] >= PROB_THRESHOLD:
                artifact_idx_prob.append(comp_idx)
                break
    eog_indices_veog, eog_indices_heog = [], []
    try:
        if "VEOG" in raw.ch_names:
            eog_indices_veog, _ = ica.find_bads_eog(raw, ch_name="VEOG", threshold=2.0, verbose=False)
        if "HEOG" in raw.ch_names:
            eog_indices_heog, _ = ica.find_bads_eog(raw, ch_name="HEOG", threshold=2.0, verbose=False)
    except Exception as e:
        print(f"  find_bads_eog failed: {e}")
    BRAIN_PROB_CEILING = 0.5
    BRAIN_CLASS_COL = ARTIFACT_CLASSES_IDX["brain"]
    eog_only_idx = (set(eog_indices_veog) | set(eog_indices_heog)) - (set(artifact_idx_argmax) | set(artifact_idx_prob))
    eog_only_rescued = {idx for idx in eog_only_idx if proba_matrix[idx, BRAIN_CLASS_COL] >= BRAIN_PROB_CEILING}
    artifact_idx = sorted(
        (set(artifact_idx_argmax) | set(artifact_idx_prob) | set(eog_indices_veog) | set(eog_indices_heog))
        - eog_only_rescued
    )
    ica.exclude = artifact_idx
    if len(ica.exclude) > 0:
        ica.apply(raw_filtered, verbose=False)
        print(f"Removed {len(ica.exclude)} ICA components: {artifact_idx}")
    else:
        print("Removed 0 ICA components")

    channels_to_drop = [ch for ch in raw_filtered.ch_names if ch.upper() in ["VEOG", "HEOG", "TP9"]]
    if channels_to_drop:
        raw_filtered.drop_channels(channels_to_drop)
        print(f"Dropped channels: {channels_to_drop}")

    return raw_filtered


def reprocess_subject_with_forced_bad(subject_id, forced_bad_channels):
    print(f"\n{'='*90}\nReprocessing {subject_id} (forced bad: {forced_bad_channels}) "
          f"-- does not affect any other subject\n{'='*90}")

    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    raw_eeg, events, sfreq_eeg, eeg_mnt_struct = load_eeg_data(eeg_folder)
    fnirs_data, fnirs_events_time, fnirs_events_code = load_and_preprocess_fnirs(subject_id)
    fnirs_data = (fnirs_data - fnirs_data.mean(axis=0, keepdims=True)) / (fnirs_data.std(axis=0, keepdims=True) + 1e-8)

    raw_eeg_clean = preprocess_eeg_force_bad(raw_eeg, events, forced_bad_channels)

    sub_out_dir = os.path.join(OUTPUT_PATH, subject_id)
    os.makedirs(sub_out_dir, exist_ok=True)
    raw_eeg_clean.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_clean_raw.fif"), overwrite=True)
    print(f"Saved cleaned EEG to {sub_out_dir}\\{subject_id}_eeg_clean_raw.fif")

    # CHANGED (single-band alignment): the old 5-band decomposition + spatial_mapping_multiband()
    # block is replaced with the SAME single-band spatial mapping used in preprocessing_v2_fixed.py --
    # one spatial_mapping() call on the full continuous, already-cleaned (1-40Hz, ICA'd) signal,
    # followed by the same per-subject z-score normalization (mean/std over the time axis, axis=0).
    eeg_mnt = get_eeg_mnt(raw_eeg_clean, eeg_mnt_struct)                       # CHANGED: no band_signals/band_data_list needed
    eeg_full_data = raw_eeg_clean.get_data()                                    # (n_channels, n_times)
    eeg_spatial_full = spatial_mapping(eeg_full_data.T, eeg_mnt)                # CHANGED: single spatial_mapping() call, matches preprocessing_v2_fixed.py exactly -> (n_times, 16, 16)
    eeg_spatial_full = (eeg_spatial_full - eeg_spatial_full.mean(axis=0, keepdims=True)) / (
        eeg_spatial_full.std(axis=0, keepdims=True) + 1e-8)                     # per-subject z-score, same as preprocessing_v2_fixed.py

    eeg_windows, fnirs_windows, labels, session_ids, trial_ids = extract_synchronized_windows(
        raw_eeg_clean, eeg_spatial_full, fnirs_data, events, sfreq_eeg, fnirs_events_time, fnirs_events_code
    )

    if len(labels) == 0:
        print(f"No windows extracted for {subject_id} -- ABORTING save (old files left untouched).")
        return

    np.save(os.path.join(sub_out_dir, f"{subject_id}_eeg_windows.npy"), eeg_windows)
    np.save(os.path.join(sub_out_dir, f"{subject_id}_fnirs_windows.npy"), fnirs_windows)
    np.save(os.path.join(sub_out_dir, f"{subject_id}_labels.npy"), labels)
    np.save(os.path.join(sub_out_dir, f"{subject_id}_session_ids.npy"), session_ids)
    np.save(os.path.join(sub_out_dir, f"{subject_id}_trial_ids.npy"), trial_ids)

    print(f"\n{subject_id} reprocessing complete!")
    print(f"EEG windows: {eeg_windows.shape}")  # CHANGED: now (N, 16, 16, 600), matching preprocessing_v2_fixed.py -- was (N, 16, 16, 5, 600)
    print(f"fNIRS windows: {fnirs_windows.shape}")
    print(f"Labels: {np.bincount(labels)}")


# ----------------------------------------------------------------------------
# RUN for all 4 target subjects
# ----------------------------------------------------------------------------
for subject_id, forced_bad in FORCED_BAD_CHANNELS.items():
    try:
        reprocess_subject_with_forced_bad(subject_id, forced_bad)
    except Exception as e:
        print(f"\n{subject_id}: FAILED -- {e}")
        import traceback
        traceback.print_exc()

print(f"\n\n{'='*90}")
print("All 4 subjects reprocessed. Next step -- verify with:")
print('  run_qc_batch(subjects=["VP015","VP018","VP019","VP020"], out=r"...")')
print(f"{'='*90}")



Reprocessing VP015 (forced bad: ['Fp1', 'Fp2']) -- does not affect any other subject
Preprocessing EEG (forcing ['Fp1', 'Fp2'] as bad -> interpolated before ICA)...
Bad channels (auto-detected + forced): ['Cz', 'CP1', 'P7', 'Fp1', 'Fp2']
Fitting ICA to data using 28 channels (please be patient, this may take a while)
Selecting by explained variance: 6 components
Fitting ICA took 14.2s.
Removed 2 ICA components: [0, 2]
Dropped channels: ['HEOG', 'VEOG']
Overwriting existing file.
Writing F:\preprocessing-history-thesis\test0.0\VP015\VP015_eeg_clean_raw.fif
Overwriting existing file.
Closing F:\preprocessing-history-thesis\test0.0\VP015\VP015_eeg_clean_raw.fif
[done]
Saved cleaned EEG to F:\preprocessing-history-thesis\test0.0\VP015\VP015_eeg_clean_raw.fif

VP015 reprocessing complete!
EEG windows: (1021, 16, 16, 600)
fNIRS windows: (1021, 11, 16, 16, 30, 2)
Labels: [342 342 337]

Reprocessing VP018 (forced bad: ['Fp1', 'Fp2']) -- does not affect any other subject
Preprocessing EEG (for

In [6]:
"""
Run D0.1 — Sanity check با ERP کلاسیک
FULLY SELF-CONTAINED CELL -- کل این محتوا را در یک سلول جدید Jupyter paste و اجرا کنید.

هدف: grand-average ERP را برای چند کانال فرونتال/پاریتال (Fz, Pz, Cz) جدا برای
سه کلاس 0/2/3-back رسم می‌کند -- دقیقاً روی سیگنال پیش‌پردازش‌شده اما قبل از
spatial mapping، یعنی مستقیم از {subject}_eeg_clean_raw.fif که preprocessing_v2_fixed.py
تولید کرده (سیگنال کانال-زمان، بعد از notch+bandpass+bad-channel-interpolation+
average-reference+ICA، قبل از هرگونه تبدیل به گرید 16x16).

معیار تصمیم (طبق پروتکل):
  - اگر حتی در averaged ERP هم تفاوت واضح بین سه کلاس دیده نشود -> مشکل از سطح
    سیگنال/برچسب است، نه معماری؛ برو سراغ Run D0.2 و D0.3 قبل از هر تغییر معماری.
  - اگر تفاوت در ERP averaged دیده می‌شود ولی مدل نمی‌گیردش -> مشکل واقعاً
    معماری/windowing است.

نیازی به GPU نیست -- فقط matplotlib/mne روی CPU، چند دقیقه طول می‌کشد.

پیش‌نیاز: preprocessing_v2_fixed.py باید در همان پوشه (یا PYTHONPATH) باشد، چون
از آن BASE_DATASET_PATH / OUTPUT_PATH / load_eeg_data / EEG_TRIAL_LABEL_MAP را
import می‌کنیم تا مسیرها و کدهای مارکر دقیقاً همان چیزی باشد که خود pipeline
استفاده می‌کند (تک منبع حقیقت، بدون ریسک ناهماهنگی/تایپوی مسیر).
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import mne

# CHANGED: import مستقیم از preprocessing_v2_fixed.py -- ایمن است چون آن فایل
# کل pipeline اصلی‌اش پشت `if __name__ == "__main__":` است، پس import فقط
# BASE_DATASET_PATH / OUTPUT_PATH / load_eeg_data / EEG_TRIAL_LABEL_MAP را
# تعریف می‌کند و چیزی اجرا نمی‌شود.
try:
    from preproc-test0 import (
        BASE_DATASET_PATH, OUTPUT_PATH, load_eeg_data, EEG_TRIAL_LABEL_MAP,
    )
except ImportError as e:
    raise ImportError(
        "Could not import from preprocessing_v2_fixed.py. Make sure it is in the "
        "same folder as this script, or on your PYTHONPATH."
    ) from e

# ============================================================================
# تنظیمات قابل‌ویرایش
# ============================================================================
SUBJECTS = [f"VP{n:03d}" for n in range(1, 27)]  # می‌توانید محدودتر کنید، مثلاً [f"VP{n:03d}" for n in range(1, 9)] برای تست سریع‌تر
CHANNELS = ['Fz', 'Pz', 'Cz']    # کانال‌های کلاسیک فرونتال/پاریتال/سنترال برای ERP
TMIN, TMAX = -1.0, 6.0           # ثانیه، نسبت به شروع trial (لحظه‌ی نمایش دستور 0/2/3-back)
                                  # پوشش: 1s قبل از دستور (baseline) تا 6s بعد از دستور
                                  # (یعنی 2s دستور + 4s اول دوره‌ی تسک -- جایی که تفاوت
                                  # آماده‌سازی/CNV بین سطوح بار شناختی بیشتر انتظار می‌رود)
BASELINE = (-1.0, 0.0)           # اصلاح baseline نسبت به قبل از شروع دستور
CLASS_NAMES = {0: '0-back', 1: '2-back', 2: '3-back'}
CLASS_COLORS = {0: 'tab:blue', 1: 'tab:orange', 2: 'tab:red'}
# ============================================================================

evokeds_by_class = {0: [], 1: [], 2: []}   # هر عضو یک mne.Evoked (میانگین یک subject) است
trial_counts_by_class = {0: 0, 1: 0, 2: 0}
n_subjects_used = 0

for subject_id in SUBJECTS:
    fif_path = os.path.join(OUTPUT_PATH, subject_id, f"{subject_id}_eeg_clean_raw.fif")
    if not os.path.exists(fif_path):
        print(f"  [SKIP] {subject_id}: {subject_id}_eeg_clean_raw.fif پیدا نشد")
        continue

    eeg_folder = os.path.join(BASE_DATASET_PATH, f"{subject_id}-EEG")
    if not os.path.exists(eeg_folder):
        print(f"  [SKIP] {subject_id}: پوشه‌ی EEG خام پیدا نشد (برای مارکرها لازم است)")
        continue

    print(f"Processing {subject_id}...")

    # سیگنال پیش‌پردازش‌شده‌ی کامل (notch+bandpass+bad-ch-interp+CAR+ICA)، قبل از spatial mapping
    raw_clean = mne.io.read_raw_fif(fif_path, preload=True, verbose=False)

    picks_available = [ch for ch in CHANNELS if ch in raw_clean.ch_names]
    if not picks_available:
        print(f"  [SKIP] {subject_id}: هیچ‌کدام از {CHANNELS} بعد از پیش‌پردازش موجود نیست")
        continue

    # مارکرهای trial را دوباره از فایل خام .mat می‌خوانیم (load_eeg_data فقط بارگذاری/
    # resample انجام می‌دهد، فیلتر/ICA نمی‌زند -- سبک و سریع است). raw برگشتی اینجا دور
    # ریخته می‌شود، فقط events لازم است. مهم: چون load_eeg_data همان resample به 200Hz
    # را قبل از محاسبه‌ی sample-index مارکرها انجام می‌دهد، این ایندکس‌ها دقیقاً روی
    # همان محور زمانی raw_clean.fif (که هم 200Hz است و هیچ crop طولی نداشته) معتبرند.
    _, events, sfreq_loaded, _ = load_eeg_data(eeg_folder)

    valid_codes = list(EEG_TRIAL_LABEL_MAP.keys())  # [112, 128, 144]
    target_events = events[np.isin(events[:, 2], valid_codes)]
    if len(target_events) == 0:
        print(f"  [SKIP] {subject_id}: هیچ مارکر trial-level پیدا نشد")
        continue

    event_id = {str(code): code for code in valid_codes}
    try:
        epochs = mne.Epochs(
            raw_clean, target_events, event_id=event_id,
            tmin=TMIN, tmax=TMAX, baseline=BASELINE,
            picks=picks_available, preload=True, reject=None, verbose=False,
        )
    except Exception as e:
        print(f"  [SKIP] {subject_id}: خطا در ساخت epochs: {e}")
        continue

    subject_had_any = False
    for code, label in EEG_TRIAL_LABEL_MAP.items():
        key = str(code)
        if key not in epochs.event_id or len(epochs[key]) == 0:
            continue
        evk = epochs[key].average()
        evokeds_by_class[label].append(evk)
        trial_counts_by_class[label] += len(epochs[key])
        subject_had_any = True

    if subject_had_any:
        n_subjects_used += 1

print(f"\n{n_subjects_used} subject استفاده شد.")
print("تعداد کل trialهای جمع‌شده به ازای هر کلاس:")
for label, name in CLASS_NAMES.items():
    print(f"  {name}: {trial_counts_by_class[label]} trial از {len(evokeds_by_class[label])} subject")

if all(len(v) == 0 for v in evokeds_by_class.values()):
    raise RuntimeError(
        "هیچ evoked ای برای هیچ کلاسی ساخته نشد -- مسیرها یا فایل‌های .fif را چک کنید."
    )

# ── Grand average روی هر کلاس (میانگین وزن‌دار خودکار mne بر اساس تعداد trial هر subject) ──
grand_avg_by_class = {}
for label in CLASS_NAMES:
    if len(evokeds_by_class[label]) > 0:
        grand_avg_by_class[label] = mne.grand_average(evokeds_by_class[label])

# ── رسم: یک subplot به ازای هر کانال، سه کلاس روی هم ──
fig, axes = plt.subplots(len(CHANNELS), 1, figsize=(10, 3.2 * len(CHANNELS)), sharex=True)
if len(CHANNELS) == 1:
    axes = [axes]

for ax, ch_name in zip(axes, CHANNELS):
    for label, name in CLASS_NAMES.items():
        if label not in grand_avg_by_class:
            continue
        evk = grand_avg_by_class[label]
        if ch_name not in evk.ch_names:
            continue
        ch_idx = evk.ch_names.index(ch_name)
        data_uv = evk.data[ch_idx] * 1e6   # ولت -> میکروولت برای خوانایی
        ax.plot(evk.times, data_uv, color=CLASS_COLORS[label], linewidth=1.6, label=name)

    ax.axvline(0.0, color='k', linestyle='--', linewidth=0.8, label='شروع دستور (t=0)')
    ax.axvline(2.0, color='gray', linestyle=':', linewidth=0.8, label='شروع دوره‌ی تسک (t=2s)')
    ax.axhline(0.0, color='k', linewidth=0.5)
    ax.set_title(f"Grand-average ERP -- کانال {ch_name}")
    ax.set_ylabel("Amplitude (µV)")
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel("Time (s) نسبت به شروع دستور")
plt.tight_layout()
plt.savefig("run_d01_grand_average_erp.png", dpi=150)
plt.show()
print("\nنمودار در run_d01_grand_average_erp.png هم ذخیره شد.")

# ── جمع‌بندی عددی سریع: میانگین دامنه در یک بازه‌ی زمانی مشخص، برای مقایسه‌ی کمّی سه کلاس ──
SUMMARY_TMIN, SUMMARY_TMAX = 2.0, 6.0  # همان بازه‌ی اول دوره‌ی تسک
print(f"\nمیانگین دامنه‌ی هر کانال در بازه‌ی [{SUMMARY_TMIN}, {SUMMARY_TMAX}]s (µV):")
for ch_name in CHANNELS:
    row = f"  {ch_name:>4}: "
    for label, name in CLASS_NAMES.items():
        if label not in grand_avg_by_class or ch_name not in grand_avg_by_class[label].ch_names:
            row += f"{name}=N/A  "
            continue
        evk = grand_avg_by_class[label]
        ch_idx = evk.ch_names.index(ch_name)
        mask = (evk.times >= SUMMARY_TMIN) & (evk.times <= SUMMARY_TMAX)
        mean_uv = evk.data[ch_idx, mask].mean() * 1e6
        row += f"{name}={mean_uv:+.2f}  "
    print(row)

SyntaxError: invalid syntax (1525268201.py, line 35)